# Synthetic Data Generation — Part 1: Prepare & Verify

**Step 0:** Parse raw KPI signals once, identify training fault windows and training Normal windows.
**Step 1:** Recompute Config A from raw signals and confirm it matches `train_X[:, :64]` exactly.

Run cell by cell. Do not continue if a check fails.

In [9]:
# =============================================================
# CELL 1 - Imports & paths
# =============================================================
import os, re, ast, json
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

CSV_PATH = "/home/jovyan/kpi_rag/data/telecomts.csv"
BASE     = "/home/jovyan/kpi_rag/Models Improvement"
DATA_DIR = BASE + "/Data Splits"
SYN_DIR  = BASE + "/Synthetic"
os.makedirs(SYN_DIR, exist_ok=True)

NUMERICAL_CHANNELS = ["RSRP", "DL_BLER", "DL_MCS", "UL_BLER", "UL_MCS",
    "UL_NPRB", "UL_SNR", "TX_Bytes", "RX_Bytes", "Estimated_UL_Buffer",
    "PRBs_DL_Current", "PRBs_UL_Current", "PRB_Utilization_DL",
    "PRB_Utilization_UL", "UL_NumberOfPackets", "DL_NumberOfPackets"]
NORMAL_LABEL = "Normal"
print("Paths set. Outputs will be saved in:", SYN_DIR)


Paths set. Outputs will be saved in: /home/jovyan/kpi_rag/Models Improvement/Synthetic


In [10]:
# =============================================================
# CELL 2 - Parse raw signals ONCE and cache them (Step 0)
# =============================================================
# Parsing 32,000 rows is slow (~5-10 min), so we save the result.
# If the cache already exists, we load it instead of re-parsing.

RAW_PATH  = os.path.join(SYN_DIR, "raw_kpis.npy")        # (32000, 16, 128)
META_PATH = os.path.join(SYN_DIR, "raw_meta.csv")        # label + anomaly segment

def parse_kpi(cell):
    clean = cell.replace("array(", "")
    clean = re.sub(r",\s*dtype=[a-zA-Z0-9]+", "", clean)
    clean = clean.replace(")", "")
    return ast.literal_eval(clean)

def parse_anomaly(cell):
    s = str(cell)
    exists = re.search(r"'exists':\s*(True|False)", s)
    exists = exists is not None and exists.group(1) == "True"
    if not exists:
        return NORMAL_LABEL, -1, -1, ""
    t  = re.search(r"'type':\s*'([^']*)'", s)
    st = re.search(r"'start':\s*(\d+)", s)
    en = re.search(r"'end':\s*(\d+)", s)
    ak = re.search(r"'affected_kpis':\s*array\(\[(.*?)\]", s, re.S)
    kpis = ";".join(re.findall(r"'([^']+)'", ak.group(1))) if ak else ""
    return (t.group(1) if t else "UNKNOWN",
            int(st.group(1)) if st else -1,
            int(en.group(1)) if en else -1,
            kpis)

if os.path.exists(RAW_PATH) and os.path.exists(META_PATH):
    raw_kpis = np.load(RAW_PATH)
    meta = pd.read_csv(META_PATH)
    print("Loaded cached files.")
else:
    df = pd.read_csv(CSV_PATH)
    n = len(df)
    raw_kpis = np.zeros((n, len(NUMERICAL_CHANNELS), 128), dtype=np.float32)
    for i, cell in enumerate(df["KPIs"]):
        d = parse_kpi(cell)
        for c, ch in enumerate(NUMERICAL_CHANNELS):
            raw_kpis[i, c, :] = np.asarray(d[ch], dtype=np.float32)
        if i % 5000 == 0:
            print(f"  parsed {i}/{n}")
    rows = [parse_anomaly(a) for a in df["anomalies"]]
    meta = pd.DataFrame(rows, columns=["label", "anom_start", "anom_end", "affected_kpis"])
    np.save(RAW_PATH, raw_kpis)
    meta.to_csv(META_PATH, index=False)
    print("Parsed and cached.")

print("raw_kpis:", raw_kpis.shape)
print("NaN values:", int(np.isnan(raw_kpis).sum()))
print("\nLabel counts (full dataset):")
print(meta["label"].value_counts())


  parsed 0/32000
  parsed 5000/32000
  parsed 10000/32000
  parsed 15000/32000
  parsed 20000/32000
  parsed 25000/32000
  parsed 30000/32000
Parsed and cached.
raw_kpis: (32000, 16, 128)
NaN values: 0

Label counts (full dataset):
label
Normal                                       30765
Jamming                                        279
Co-Channel Interference (Mild)                 146
Buffer Overflow (Gradual Buildup)              138
Co-Channel Interference (Severe)               114
Antenna Failure                                107
Faulty RF Filters (Temporal)                   100
High Network Congestion (Gradual Buildup)       93
Doppler Shift (Severe)                          86
Faulty Handover Algorithm (Too Frequent)        64
Resource Allocation Bugs                        60
High Network Congestion (Sudden Spike)          48
Name: count, dtype: int64


In [12]:
# =============================================================
# CELL 3 - Training partition: fault windows + Normal windows (Step 0)
# =============================================================
train_X     = np.load(os.path.join(DATA_DIR, "train_X_sessionsplit_fixed.npy"), allow_pickle=True)
train_y_str = np.load(os.path.join(DATA_DIR, "train_y_sessionsplit.npy"), allow_pickle=True)
si_train    = np.load(os.path.join(DATA_DIR, "train_idx_sessionsplit.npy"), allow_pickle=True).astype(int)
si_test     = np.load(os.path.join(DATA_DIR, "test_idx_sessionsplit.npy"), allow_pickle=True).astype(int)
session_id_by_row = np.load(os.path.join(DATA_DIR, "session_id_by_row.npy"), allow_pickle=True)

# --- Check 1: row order. CSV label at si_train[i] must equal train_y[i] ---
csv_labels_train = meta["label"].values[si_train]
order_ok = np.mean(csv_labels_train == train_y_str.astype(str))
print(f"Label agreement CSV vs train_y: {order_ok*100:.2f}%  (must be 100%)")
assert order_ok == 1.0, "Row order mismatch - stop and investigate."

# --- Check 2: no test sessions leak into training ---
train_sessions = session_id_by_row[si_train]
test_sessions  = session_id_by_row[si_test]
overlap = set(np.unique(train_sessions)) & set(np.unique(test_sessions))
print("Train/test session overlap:", overlap if overlap else "none")
assert not overlap

# --- The two groups we work with ---
is_fault  = train_y_str != NORMAL_LABEL
fault_rows  = si_train[is_fault]      # original CSV row indices
normal_rows = si_train[~is_fault]

print(f"\nTraining fault windows : {len(fault_rows)}  (expected 935)")
print(f"Training Normal windows: {len(normal_rows)}")
print(f"Training sessions      : {len(np.unique(train_sessions))}")
print(f"Sessions containing faults: {len(np.unique(train_sessions[is_fault]))}")
print(f"Sessions with Normal windows: {len(np.unique(train_sessions[~is_fault]))}")


Label agreement CSV vs train_y: 100.00%  (must be 100%)
Train/test session overlap: none

Training fault windows : 935  (expected 935)
Training Normal windows: 24753
Training sessions      : 28
Sessions containing faults: 10
Sessions with Normal windows: 19


In [13]:
# =============================================================
# CELL 4 - Verify Config A reconstruction (Step 1) — CORRECT
# =============================================================
import re

WINDOW_SIZE = 128

def extract_kpi_array(kpi_text, channel_name):
    pattern = rf"'{channel_name}':\s*array\(\[(.*?)\]"
    match = re.search(pattern, str(kpi_text), re.DOTALL)
    if not match:
        return np.zeros(WINDOW_SIZE)
    values_str = match.group(1)
    values = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', values_str)
    arr = np.array([float(v) for v in values[:WINDOW_SIZE]])
    if len(arr) < WINDOW_SIZE:
        arr = np.pad(arr, (0, WINDOW_SIZE - len(arr)))
    return arr

def compute_config_A_from_text(kpi_text):
    features = []
    for channel in NUMERICAL_CHANNELS:
        arr = extract_kpi_array(kpi_text, channel)
        features.extend([np.mean(arr), np.std(arr), np.min(arr), np.max(arr)])
    return np.array(features, dtype=np.float32)

# Test on first training window
import pandas as pd
df_csv = pd.read_csv(CSV_PATH)
test_row = df_csv["KPIs"].iloc[si_train[0]]
computed = compute_config_A_from_text(test_row)
actual = train_X[0, :64]

max_err = float(np.max(np.abs(computed - actual) / (np.abs(actual) + 1e-9)))
print(f"Max relative error: {max_err:.2e}")
assert max_err < 1e-3, "Config A mismatch — stop."
print("Step 1 PASSED — Config A formula confirmed.")

with open(os.path.join(SYN_DIR, "config_A_spec.json"), "w") as f:
    json.dump({"source": "mean_std_min_max",
               "channels": NUMERICAL_CHANNELS,
               "ddof": 0}, f, indent=2)
print("Saved config_A_spec.json")

Max relative error: 5.39e-08
Step 1 PASSED — Config A formula confirmed.
Saved config_A_spec.json


## Part 2 — Estimate Fault Recipes

For each of the 10 synthetic fault types, we learn from the real training windows:
- Which KPIs change during the fault
- The direction and magnitude of the change (compared to Normal windows)

This gives us a "recipe" per fault — used in Part 3 to inject faults into Normal windows.
Jamming is excluded (real fault, handled by augmentation in Part 3).

In [14]:
# =============================================================
# CELL 6 - Compute per-fault KPI change statistics (Step 2)
# =============================================================
import pandas as pd, json

df_csv = pd.read_csv(CSV_PATH)

SYNTHETIC_FAULTS = [
    "Co-Channel Interference (Mild)",
    "Co-Channel Interference (Severe)",
    "Buffer Overflow (Gradual Buildup)",
    "Antenna Failure",
    "Faulty RF Filters (Temporal)",
    "High Network Congestion (Gradual Buildup)",
    "Doppler Shift (Severe)",
    "Faulty Handover Algorithm (Too Frequent)",
    "Resource Allocation Bugs",
    "High Network Congestion (Sudden Spike)",
]

# Normal windows mean per channel (from training only)
normal_mask = train_y_str == NORMAL_LABEL
normal_rows_csv = si_train[normal_mask]
normal_X = train_X[normal_mask][:, :64]  # Config A

# Normal mean and std per feature
normal_mean = normal_X.mean(axis=0)
normal_std  = normal_X.std(axis=0)

print(f"Normal windows used for reference: {len(normal_rows_csv)}")
print(f"Normal mean shape: {normal_mean.shape}")

# Per-fault statistics
recipes = {}

for fault in SYNTHETIC_FAULTS:
    fault_mask = train_y_str == fault
    fault_X = train_X[fault_mask][:, :64]
    n = fault_X.shape[0]
    
    if n == 0:
        print(f"WARNING: {fault} has 0 training windows — skipping")
        continue
    
    fault_mean = fault_X.mean(axis=0)
    fault_std  = fault_X.std(axis=0)
    
    # Z-score: how many stds away from Normal mean
    z_score = (fault_mean - normal_mean) / (normal_std + 1e-9)
    
    # Find top-5 most discriminative channels
    abs_z = np.abs(z_score)
    top_features = np.argsort(abs_z)[::-1][:10]
    
    # Build recipe
    feature_names = [f"{ch}_{st}" 
                     for ch in NUMERICAL_CHANNELS 
                     for st in ["mean", "std", "min", "max"]]
    
    recipe = {
        "n_windows": int(n),
        "top_features": [
            {
                "feature": feature_names[i],
                "channel": NUMERICAL_CHANNELS[i // 4],
                "stat": ["mean", "std", "min", "max"][i % 4],
                "z_score": float(z_score[i]),
                "fault_mean": float(fault_mean[i]),
                "normal_mean": float(normal_mean[i]),
                "direction": "increase" if z_score[i] > 0 else "decrease",
                "magnitude_pct": float((fault_mean[i] - normal_mean[i]) / 
                                       (abs(normal_mean[i]) + 1e-9) * 100)
            }
            for i in top_features
        ]
    }
    recipes[fault] = recipe
    
    print(f"\n{fault} (n={n})")
    print(f"  Top 3 discriminative features:")
    for feat in recipe["top_features"][:3]:
   SYN_DIR  = "/home/jovyan/kpi_rag/Models Improvement/Synthetic"
DATA_DIR = "/home/jovyan/kpi_rag/Models Improvement/Data Splits"
CSV_PATH = "/home/jovyan/kpi_rag/data/telecomts.csv"     print(f"    {feat['feature']:35s} z={feat['z_score']:+.2f}  "
              f"({feat['direction']}, {feat['magnitude_pct']:+.1f}%)")

with open(os.path.join(SYN_DIR, "fault_recipes.json"), "w") as f:
    json.dump(recipes, f, indent=2)
print("\nSaved fault_recipes.json")

Normal windows used for reference: 24753
Normal mean shape: (64,)

Co-Channel Interference (Mild) (n=126)
  Top 3 discriminative features:
    Estimated_UL_Buffer_std             z=+3.10  (increase, +566.4%)
    UL_SNR_min                          z=-1.66  (decrease, -21.2%)
    Estimated_UL_Buffer_max             z=+1.31  (increase, +281.2%)

Co-Channel Interference (Severe) (n=84)
  Top 3 discriminative features:
    UL_SNR_mean                         z=-8.76  (decrease, -37.7%)
    UL_SNR_min                          z=-5.81  (decrease, -73.9%)
    UL_SNR_std                          z=+2.87  (increase, +188.2%)

Buffer Overflow (Gradual Buildup) (n=117)
  Top 3 discriminative features:
    PRB_Utilization_UL_min              z=+24.60  (increase, +3467.9%)
    Estimated_UL_Buffer_std             z=+22.52  (increase, +4121.8%)
    Estimated_UL_Buffer_mean            z=+20.00  (increase, +4130.4%)

Antenna Failure (n=98)
  Top 3 discriminative features:
    DL_MCS_std                

## Part 3 — Synthetic Window Generator

For each synthetic fault:
1. Pick a random Normal window from training sessions
2. Apply the fault recipe: scale the most discriminative KPI channels
3. Clip values to physical limits
4. Assign the fault label and the source session ID

Jamming: augment real Jamming windows with jitter + magnitude scaling.

In [16]:
import pandas as pd, json, numpy as np, os

SYN_DIR  = "/home/jovyan/kpi_rag/Models Improvement/Synthetic"
DATA_DIR = "/home/jovyan/kpi_rag/Models Improvement/Data Splits"
CSV_PATH = "/home/jovyan/kpi_rag/data/telecomts.csv"
WINDOW_SIZE = 128
SEED = 42

NUMERICAL_CHANNELS = ["RSRP", "DL_BLER", "DL_MCS", "UL_BLER", "UL_MCS",
    "UL_NPRB", "UL_SNR", "TX_Bytes", "RX_Bytes", "Estimated_UL_Buffer",
    "PRBs_DL_Current", "PRBs_UL_Current", "PRB_Utilization_DL",
    "PRB_Utilization_UL", "UL_NumberOfPackets", "DL_NumberOfPackets"]

PHYS_LIMITS = {
    "RSRP": (-140,-45), "DL_BLER": (0,1), "DL_MCS": (0,27),
    "UL_BLER": (0,1), "UL_MCS": (0,27), "UL_NPRB": (0,100),
    "UL_SNR": (-20,40), "TX_Bytes": (0,None), "RX_Bytes": (0,None),
    "Estimated_UL_Buffer": (0,None), "PRBs_DL_Current": (0,100),
    "PRBs_UL_Current": (0,100), "PRB_Utilization_DL": (0,100),
    "PRB_Utilization_UL": (0,100), "UL_NumberOfPackets": (0,None),
    "DL_NumberOfPackets": (0,None),
}

import re

def extract_kpi_array(kpi_text, channel_name):
    pattern = rf"'{channel_name}':\s*array\(\[(.*?)\]"
    match = re.search(pattern, str(kpi_text), re.DOTALL)
    if not match:
        return np.zeros(WINDOW_SIZE)
    values = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', match.group(1))
    arr = np.array([float(v) for v in values[:WINDOW_SIZE]])
    if len(arr) < WINDOW_SIZE:
        arr = np.pad(arr, (0, WINDOW_SIZE - len(arr)))
    return arr

def clip_channel(arr, ch):
    lo, hi = PHYS_LIMITS.get(ch, (None, None))
    return np.clip(arr, lo, hi)

def signals_to_config_A(signals):
    features = []
    for ch in NUMERICAL_CHANNELS:
        arr = signals[ch]
        features.extend([np.mean(arr), np.std(arr), np.min(arr), np.max(arr)])
    return np.array(features, dtype=np.float32)

def inject_fault(normal_kpi_text, recipe, rng):
    signals = {ch: extract_kpi_array(normal_kpi_text, ch).copy() 
               for ch in NUMERICAL_CHANNELS}
    for feat_info in recipe["top_features"][:5]:
        ch = feat_info["channel"]
        z  = feat_info["z_score"]
        signal_std = np.std(signals[ch])
        abs_z  = min(abs(z), 5.0)
        scale  = rng.uniform(0.3, 0.7)
        delta  = np.sign(z) * abs_z * signal_std * scale
        train_X = np.load(os.path.join(DATA_DIR, "train_X_sessionsplit_fixed.npy"), allow_pickle=True)
train_y_str = np.load(os.path.join(DATA_DIR, "train_y_sessionsplit.npy"), allow_pickle=True)
si_train = np.load(os.path.join(DATA_DIR, "train_idx_sessionsplit.npy"), allow_pickle=True).astype(int)
session_id_by_row = np.load(os.path.join(DATA_DIR, "session_id_by_row.npy"), allow_pickle=True)

signals[ch] = signals[ch] + delta
        signals[ch] += rng.normal(0, signal_std * 0.05, WINDOW_SIZE)
        signals[ch]  = clip_channel(signals[ch], ch)
    return signals

def augment_jamming(kpi_text, rng):
    signals = {}
    for ch in NUMERICAL_CHANNELS:
        arr = extract_kpi_array(kpi_text, ch).copy()
        arr = arr * rng.uniform(0.95, 1.05)
        arr += rng.normal(0, np.std(arr) * 0.03, len(arr))
        signals[ch] = clip_channel(arr, ch)
    return signals

# Load required data
rng = np.random.default_rng(SEED)
df_csv = pd.read_csv(CSV_PATH)
train_X = np.load(os.path.join(DATA_DIR, "train_X_sessionsplit_fixed.npy"), allow_pickle=True)
train_y_str = np.load(os.path.join(DATA_DIR, "train_y_sessionsplit.npy"), allow_pickle=True)
si_train = np.load(os.path.join(DATA_DIR, "train_idx_sessionsplit.npy"), allow_pickle=True).astype(int)
session_id_by_row = np.load(os.path.join(DATA_DIR, "session_id_by_row.npy"), allow_pickle=True)


NORMAL_LABEL = "Normal"
normal_mask = train_y_str == NORMAL_LABEL
normal_rows = si_train[normal_mask]
normal_kpi_texts   = df_csv["KPIs"].values[normal_rows]
normal_session_ids = session_id_by_row[normal_rows]

jamming_mask = train_y_str == "Jamming"
jamming_rows_csv    = si_train[jamming_mask]
jamming_kpi_texts   = df_csv["KPIs"].values[jamming_rows_csv]
jamming_session_ids = session_id_by_row[jamming_rows_csv]

with open(os.path.join(SYN_DIR, "fault_recipes.json")) as f:
    recipes = json.load(f)

SYNTHETIC_FAULTS = list(recipes.keys())
N_TARGET = 150
fault_counts = {f: int((train_y_str == f).sum()) for f in SYNTHETIC_FAULTS}
fault_counts["Jamming"] = int((train_y_str == "Jamming").sum())

syn_features, syn_labels, syn_sessions = [], [], []

for fault in SYNTHETIC_FAULTS:
    needed = max(0, N_TARGET - fault_counts[fault])
    if needed == 0:
        continue
    recipe = recipes[fault]
    for _ in range(needed):
        idx = rng.integers(len(normal_kpi_texts))
        signals = inject_fault(normal_kpi_texts[idx], recipe, rng)
        syn_features.append(signals_to_config_A(signals))
        syn_labels.append(fault)
        syn_sessions.append(normal_session_ids[idx])

jamming_needed = max(0, N_TARGET - fault_counts["Jamming"])
if jamming_needed > 0:
    for _ in range(jamming_needed):
        idx = rng.integers(len(jamming_kpi_texts))
        signals = augment_jamming(jamming_kpi_texts[idx], rng)
        syn_features.append(signals_to_config_A(signals))
        syn_labels.append("Jamming")
        syn_sessions.append(jamming_session_ids[idx])

syn_features = np.array(syn_features, dtype=np.float32)
syn_labels   = np.array(syn_labels)
syn_sessions = np.array(syn_sessions)

np.save(os.path.join(SYN_DIR, "syn_features.npy"), syn_features)
np.save(os.path.join(SYN_DIR, "syn_labels.npy"), syn_labels)
np.save(os.path.join(SYN_DIR, "syn_sessions.npy"), syn_sessions)

print("Shape:", syn_features.shape)
print("Mean:", syn_features.mean())
print("Std:", syn_features.std())

Shape: (750, 64)
Mean: 11501.7705
Std: 67279.11


## Part 4 — Validate Synthetic Data Quality

Three checks before training:
1. Visual: plot real vs synthetic signals for 2 fault types
2. Distribution: compare Config A feature distributions
3. Realism test: can a classifier separate real from synthetic?
   If it separates them too easily → generator needs fixing.

In [18]:
# =============================================================
# CELL 10 - Validate Synthetic Data Quality (Step 4)
# =============================================================
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

syn_features = np.load(os.path.join(SYN_DIR, "syn_features.npy"))
syn_labels   = np.load(os.path.join(SYN_DIR, "syn_labels.npy"))
syn_sessions = np.load(os.path.join(SYN_DIR, "syn_sessions.npy"))

# ── Check 1: Feature distribution comparison ──────────────────
print("=== Check 1: Feature Distribution ===")
fault_mask = train_y_str != NORMAL_LABEL
real_fault_X = train_X[fault_mask][:, :64]

print(f"Real fault features   : mean={real_fault_X.mean():.3f}, std={real_fault_X.std():.3f}")
print(f"Synthetic features    : mean={syn_features.mean():.3f}, std={syn_features.std():.3f}")
print(f"Mean difference       : {abs(real_fault_X.mean() - syn_features.mean()):.4f}")

# ── Check 2: Realism test ─────────────────────────────────────
print("\n=== Check 2: Realism Test ===")
print("Training a classifier to separate real vs synthetic...")
print("(Score near 0.5 = realistic, near 1.0 = too different)")

# Take equal samples of real and synthetic
n_check = min(len(real_fault_X), len(syn_features))
X_real_s = real_fault_X[:n_check]
X_syn_s  = syn_features[:n_check]

X_check = np.vstack([X_real_s, X_syn_s])
y_check = np.array([0]*n_check + [1]*n_check)

clf = RandomForestClassifier(n_estimators=50, random_state=SEED, n_jobs=-1)
scores = cross_val_score(clf, X_check, y_check, cv=5, scoring="roc_auc")
print(f"ROC-AUC (real vs synthetic): {scores.mean():.3f} ± {scores.std():.3f}")

if scores.mean() < 0.70:
    print("VERDICT: Synthetic data looks realistic enough to use.")
elif scores.mean() < 0.85:
    print("VERDICT: Synthetic data is somewhat detectable — acceptable for this dataset size.")
else:
    print("VERDICT: Synthetic data is too different from real — consider revising recipes.")

# ── Check 3: Per-fault session diversity ──────────────────────
print("\n=== Check 3: Session Diversity ===")
print("Synthetic windows come from these sessions:")
for fault in np.unique(syn_labels):
    mask = syn_labels == fault
    sessions = np.unique(syn_sessions[mask])
    print(f"  {fault:<45s} sessions={sessions}")

print("\nAll checks complete.")

=== Check 1: Feature Distribution ===
Real fault features   : mean=7931.670, std=49961.264
Synthetic features    : mean=11501.771, std=67279.109
Mean difference       : 3570.1006

=== Check 2: Realism Test ===
Training a classifier to separate real vs synthetic...
(Score near 0.5 = realistic, near 1.0 = too different)
ROC-AUC (real vs synthetic): 0.972 ± 0.022
VERDICT: Synthetic data is too different from real — consider revising recipes.

=== Check 3: Session Diversity ===
Synthetic windows come from these sessions:
  Antenna Failure                               sessions=[ 0  5  6  7  8  9 10 22 23 25 26 27 28 29 30 31]
  Buffer Overflow (Gradual Buildup)             sessions=[ 0  5  6  7  8  9 10 20 21 23 24 25 28 30]
  Co-Channel Interference (Mild)                sessions=[ 0  5  6  8  9 10 24 28]
  Co-Channel Interference (Severe)              sessions=[ 0  5  6  7  8  9 10 20 22 24 25 27 28 31]
  Doppler Shift (Severe)                        sessions=[ 0  5  6  7  8  9 10 20 21 

## Part 5 — CV Experiment with Synthetic Data
Compare real-only vs real + synthetic under the same session-aware CV.
Synthetic windows enter training folds only — validation uses real data only.

In [21]:
# =============================================================
# CELL 12 - CV Experiment: Real only vs Real + Synthetic
# =============================================================
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier
from collections import Counter

syn_features = np.load(os.path.join(SYN_DIR, "syn_features.npy"))
syn_labels   = np.load(os.path.join(SYN_DIR, "syn_labels.npy"))
syn_sessions = np.load(os.path.join(SYN_DIR, "syn_sessions.npy"))

fault_mask    = train_y_str != NORMAL_LABEL
real_X        = train_X[fault_mask][:, :64]
real_y        = train_y_str[fault_mask]
real_sessions = session_id_by_row[si_train][fault_mask]

def run_cv(use_synthetic, label, k=5):
    # Folds are built on REAL data only
    actual_k = min(k, len(np.unique(real_sessions)))
    cv = StratifiedGroupKFold(n_splits=actual_k, shuffle=True, random_state=SEED)
    fold_f1s = []
    for fold_i, (tr_i, val_i) in enumerate(cv.split(real_X, real_y, groups=real_sessions)):
        X_tr, y_tr = real_X[tr_i], real_y[tr_i]
        X_val, y_val = real_X[val_i], real_y[val_i]      # validation = real only

        if use_synthetic:
            # Add synthetic windows whose source session is NOT in validation
            val_sessions = set(np.unique(real_sessions[val_i]))
            keep = np.array([s not in val_sessions for s in syn_sessions])
            X_tr = np.vstack([X_tr, syn_features[keep]])
            y_tr = np.concatenate([y_tr, syn_labels[keep]])

        counts  = Counter(y_tr)
        weights = {c: len(y_tr) / (len(counts) * n) for c, n in counts.items()}
        clf = CatBoostClassifier(random_seed=SEED, verbose=0, thread_count=4,
                                 class_weights=weights)
        clf.fit(X_tr, y_tr)
        preds = clf.predict(X_val).ravel()
        f1 = f1_score(y_val, preds, average="macro", zero_division=0)
        fold_f1s.append(f1)
        print(f"  Fold {fold_i+1}: {f1:.4f}")
    m, s = np.mean(fold_f1s), np.std(fold_f1s)
    print(f"\n{label}: {m:.4f} ± {s:.4f}  CI=[{m-1.96*s:.4f}, {m+1.96*s:.4f}]")
    return m, s

print("=== BASELINE: Real only ===")
base_mean, base_std = run_cv(False, "Real only")

print("\n=== EXPERIMENT: Real + Synthetic ===")
exp_mean, exp_std = run_cv(True, "Real + Synthetic")

delta = exp_mean - base_mean
print("\n" + "=" * 55)
print(f"{'Real only':<30} {base_mean:.4f} ± {base_std:.4f}")
print(f"{'Real + Synthetic':<30} {exp_mean:.4f} ± {exp_std:.4f}")
print(f"{'Delta':<30} {delta:+.4f}")
print("=" * 55)

with open(os.path.join(SYN_DIR, "synthetic_experiment_results.json"), "w") as f:
    json.dump({"baseline_mean": float(base_mean), "baseline_std": float(base_std),
               "augmented_mean": float(exp_mean), "augmented_std": float(exp_std),
               "delta": float(delta)}, f, indent=2)
print("Saved synthetic_experiment_results.json")

=== BASELINE: Real only ===
  Fold 1: 0.2132
  Fold 2: 0.2749
  Fold 3: 0.2909
  Fold 4: 0.2429
  Fold 5: 0.4491

Real only: 0.2942 ± 0.0819  CI=[0.1336, 0.4548]

=== EXPERIMENT: Real + Synthetic ===
  Fold 1: 0.1645
  Fold 2: 0.1687
  Fold 3: 0.3418
  Fold 4: 0.3305
  Fold 5: 0.4258

Real + Synthetic: 0.2863 ± 0.1031  CI=[0.0841, 0.4884]

Real only                      0.2942 ± 0.0819
Real + Synthetic               0.2863 ± 0.1031
Delta                          -0.0080
Saved synthetic_experiment_results.json


## Part 6 — Attempt 2: Improved Fault Injection (v2)

### Attempt 1 result
- CV F1-macro: Real only = 0.294 | Real + Synthetic (v1) = 0.286 (Δ = −0.008, no improvement)
- Realism test AUC = 0.97 → synthetic windows were easy to tell apart from real ones.

### Why Attempt 1 failed
1. It applied a **constant shift** to each channel. Most fault recipes are driven by
   **variability** (e.g., Estimated_UL_Buffer_std z = +22), and a constant shift does not
   change the std at all.
2. It changed the **whole window**, while real faults often occupy only part of it.
3. A channel appearing more than once in the top features was shifted more than once.

### Attempt 2 approach
- Targets are learned from the **anomaly segment** of real fault windows
  (using `anom_start` / `anom_end`), not the whole window.
- **Stat-aware injection** per channel: shift the mean AND scale the variability
  (std) to match the real fault's segment behaviour. Min/max follow naturally.
- Only **affected channels** are modified (clear mean shift or std change vs Normal).
- Robust medians instead of means (KPIs like bytes are heavy-tailed).
- Random variation of ±30% per synthetic window for diversity.
- Same target (150 per class) and same CV protocol as Attempt 1 → fair comparison.

In [22]:
# =============================================================
# CELL 14 - Learn segment-level fault targets (v2)
# =============================================================
rng = np.random.default_rng(SEED)
EPS = 1e-6

def window_signals(row):
    """Original CSV row -> (16, 128) array, using the verified extractor."""
    text = df_csv["KPIs"].values[row]
    return np.stack([extract_kpi_array(text, ch) for ch in NUMERICAL_CHANNELS])

# Normal reference from 2,000 training Normal windows
ref_rows = rng.choice(normal_rows, size=2000, replace=False)
ref = np.stack([window_signals(r) for r in ref_rows])            # (2000, 16, 128)
normal_mean_ref = np.median(ref.mean(axis=2), axis=0)             # (16,)
normal_std_ref  = np.median(ref.std(axis=2), axis=0)              # (16,)

targets = {}
for fault in SYNTHETIC_FAULTS:
    rows = si_train[train_y_str == fault]
    seg_means, seg_stds, segments = [], [], []
    for r in rows:
        x = window_signals(r)
        s, e = int(meta.loc[r, "anom_start"]), int(meta.loc[r, "anom_end"])
        if s < 0 or e < s:
            s, e = 0, 127
        seg = x[:, s:e + 1]
        seg_means.append(seg.mean(axis=1))
        seg_stds.append(seg.std(axis=1))
        segments.append([s, e])

    shift     = np.median(seg_means, axis=0) - normal_mean_ref
    seg_std   = np.median(seg_stds, axis=0)
    std_ratio = np.clip(seg_std / np.maximum(normal_std_ref, EPS), 0.2, 20)

    # Affected = clear mean shift OR clear change in variability
    shift_z  = np.abs(shift) / (normal_std_ref + EPS)
    affected = (shift_z > 0.5) | (std_ratio < 0.67) | (std_ratio > 1.5)

    targets[fault] = {"shift": shift.tolist(), "std_ratio": std_ratio.tolist(),
                      "seg_std": seg_std.tolist(), "affected": affected.tolist(),
                      "segments": segments}

    seg_len = np.median([e - s + 1 for s, e in segments])
    names = [NUMERICAL_CHANNELS[c] for c in np.where(affected)[0]]
    print(f"{fault} (n={len(rows)}) | median segment length={seg_len:.0f} | "
          f"affected channels ({len(names)}): {names}")

with open(os.path.join(SYN_DIR, "fault_targets_v2.json"), "w") as f:
    json.dump(targets, f, indent=2)
print("\nSaved fault_targets_v2.json")

Co-Channel Interference (Mild) (n=126) | median segment length=128 | affected channels (16): ['RSRP', 'DL_BLER', 'DL_MCS', 'UL_BLER', 'UL_MCS', 'UL_NPRB', 'UL_SNR', 'TX_Bytes', 'RX_Bytes', 'Estimated_UL_Buffer', 'PRBs_DL_Current', 'PRBs_UL_Current', 'PRB_Utilization_DL', 'PRB_Utilization_UL', 'UL_NumberOfPackets', 'DL_NumberOfPackets']
Co-Channel Interference (Severe) (n=84) | median segment length=104 | affected channels (16): ['RSRP', 'DL_BLER', 'DL_MCS', 'UL_BLER', 'UL_MCS', 'UL_NPRB', 'UL_SNR', 'TX_Bytes', 'RX_Bytes', 'Estimated_UL_Buffer', 'PRBs_DL_Current', 'PRBs_UL_Current', 'PRB_Utilization_DL', 'PRB_Utilization_UL', 'UL_NumberOfPackets', 'DL_NumberOfPackets']
Buffer Overflow (Gradual Buildup) (n=117) | median segment length=101 | affected channels (15): ['RSRP', 'DL_BLER', 'DL_MCS', 'UL_BLER', 'UL_MCS', 'UL_NPRB', 'TX_Bytes', 'RX_Bytes', 'Estimated_UL_Buffer', 'PRBs_DL_Current', 'PRBs_UL_Current', 'PRB_Utilization_DL', 'PRB_Utilization_UL', 'UL_NumberOfPackets', 'DL_NumberOfPa

In [23]:
# =============================================================
# CELL 15 - Synthetic generator v2 (segment-level, stat-aware)
# =============================================================
rng = np.random.default_rng(SEED)

def inject_fault_v2(normal_row, fault, rng):
    x = window_signals(normal_row).astype(np.float64)
    t = targets[fault]
    s, e = t["segments"][rng.integers(len(t["segments"]))]   # realistic fault position
    for c, ch in enumerate(NUMERICAL_CHANNELS):
        if not t["affected"][c]:
            continue
        seg = x[c, s:e + 1]
        m, sd = seg.mean(), seg.std()
        u_shift, u_std = rng.uniform(0.7, 1.3, size=2)
        shift = t["shift"][c] * u_shift
        if sd > EPS:
            # scale variability around the local mean, then shift the level
            seg_new = (seg - m) * (t["std_ratio"][c] * u_std) + m + shift
        else:
            # flat signal: create variability directly
            seg_new = seg + shift + rng.normal(0, t["seg_std"][c] * u_std, len(seg))
        x[c, s:e + 1] = clip_channel(seg_new, ch)
    return x

def array_to_config_A(x):
    """(16, 128) -> (64,): per channel [mean, std, min, max], same order as Config A."""
    return np.stack([x.mean(1), x.std(1), x.min(1), x.max(1)], axis=1).reshape(-1).astype(np.float32)

N_TARGET = 150
v2_features, v2_labels, v2_sessions = [], [], []
for fault in SYNTHETIC_FAULTS:
    needed = max(0, N_TARGET - int((train_y_str == fault).sum()))
    for _ in range(needed):
        r = normal_rows[rng.integers(len(normal_rows))]
        v2_features.append(array_to_config_A(inject_fault_v2(r, fault, rng)))
        v2_labels.append(fault)
        v2_sessions.append(session_id_by_row[r])
# Jamming already has 185 real windows (> 150), so none are generated.

v2_features = np.array(v2_features, dtype=np.float32)
v2_labels   = np.array(v2_labels)
v2_sessions = np.array(v2_sessions)

np.save(os.path.join(SYN_DIR, "syn_v2_features.npy"), v2_features)
np.save(os.path.join(SYN_DIR, "syn_v2_labels.npy"),   v2_labels)
np.save(os.path.join(SYN_DIR, "syn_v2_sessions.npy"), v2_sessions)
print("v2 shape:", v2_features.shape, "| mean:", v2_features.mean(), "| std:", v2_features.std())

v2 shape: (750, 64) | mean: 9681.869 | std: 56799.47


### Code (validation: v1 vs v2)

In [25]:
# =============================================================
# CELL 16 - Realism check, class-matched (v1 vs v2)
# =============================================================
# Fairer than Cell 10: compares real vs synthetic WITHIN the same fault
# classes, so the classifier can't just use class-mix differences.
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(SEED)
fault_mask = train_y_str != NORMAL_LABEL
real_X, real_y = train_X[fault_mask][:, :64], train_y_str[fault_mask]

for version, prefix in [("v1", "syn"), ("v2", "syn_v2")]:
    F = np.load(os.path.join(SYN_DIR, f"{prefix}_features.npy"))
    L = np.load(os.path.join(SYN_DIR, f"{prefix}_labels.npy"))
    Xr, Xs = [], []
    for fault in np.unique(L):
        r, s = real_X[real_y == fault], F[L == fault]
        n = min(len(r), len(s))
        if n < 5:
            continue
        Xr.append(r[rng.choice(len(r), n, replace=False)])
        Xs.append(s[rng.choice(len(s), n, replace=False)])
    Xr, Xs = np.vstack(Xr), np.vstack(Xs)
    Xc = np.vstack([Xr, Xs])
    yc = np.array([0] * len(Xr) + [1] * len(Xs))
    auc = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
                          Xc, yc, cv=5, scoring="roc_auc")
    print(f"{version}: class-matched realism AUC = {auc.mean():.3f} ± {auc.std():.3f}")

print("\nLower is better (0.5 = indistinguishable from real).")

v1: class-matched realism AUC = 0.994 ± 0.005
v2: class-matched realism AUC = 0.985 ± 0.017

Lower is better (0.5 = indistinguishable from real).


## Part 7 — Attempt 3: Context-Controlled Fault-Effect Transplantation (v3)

### Problem found in Attempt 2
Fault windows and Normal windows come from different recordings (only 1 of 28
training sessions contains both). A fault-vs-Normal comparison therefore mixes
the fault effect with recording context (zone, application, traffic level).

### v3 method
1. Within-window control: fault effect = anomaly segment vs the normal part of the
   SAME window (same recording, no context confound).
2. Fallback for full-window faults: compare against context-matched Normal windows
   (same zone, application, mobility, congestion).
3. Statistical gating: a channel is "affected" only if its effect is consistent
   across real windows (statistical test).
4. Effect-vector bootstrap: each synthetic window receives the full effect vector
   of one real fault window, preserving cross-channel correlations.
5. Context-matched injection: only inject into Normal windows whose context
   matches real fault windows.
6. Effects include level shift, variability ratio, and trend (slope).

In [17]:
# =============================================================
# CELL 16b - Feasibility check for v3
# =============================================================
import ast

# --- Parse context from the labels column ---
ctx = df_csv["labels"].apply(lambda s: ast.literal_eval(s) if isinstance(s, str) else {})
context = pd.DataFrame({
    "zone":       ctx.apply(lambda d: d.get("zone")),
    "app":        ctx.apply(lambda d: d.get("application")),
    "mobility":   ctx.apply(lambda d: d.get("mobility")),
    "congestion": ctx.apply(lambda d: d.get("congestion")),
})
context.to_csv(os.path.join(SYN_DIR, "context.csv"), index=False)

# --- Check 1: how many fault windows support the within-window control ---
MIN_OUTSIDE = 16   # need at least 16 normal timesteps outside the fault
print("=== Within-window control availability ===")
for fault in SYNTHETIC_FAULTS:
    rows = si_train[train_y_str == fault]
    s = meta.loc[rows, "anom_start"].values
    e = meta.loc[rows, "anom_end"].values
    usable = (s >= 0) & ((128 - (e - s + 1)) >= MIN_OUTSIDE)
    print(f"  {fault:<45s} {usable.sum():3d} / {len(rows)} usable")

# --- Check 2: contexts of fault windows vs Normal windows ---
print("\n=== Fault window contexts (zone, app, mobility, congestion) ===")
fault_rows_all = si_train[train_y_str != NORMAL_LABEL]
print(context.loc[fault_rows_all].value_counts().to_string())

print("\n=== Normal window contexts (training) ===")
print(context.loc[normal_rows].value_counts().to_string())

=== Within-window control availability ===
  Co-Channel Interference (Mild)                 53 / 126 usable
  Co-Channel Interference (Severe)               44 / 84 usable
  Buffer Overflow (Gradual Buildup)              61 / 117 usable
  Antenna Failure                                73 / 98 usable
  Faulty RF Filters (Temporal)                   41 / 67 usable
  High Network Congestion (Gradual Buildup)      29 / 56 usable
  Doppler Shift (Severe)                         48 / 77 usable
  Faulty Handover Algorithm (Too Frequent)       34 / 45 usable
  Resource Allocation Bugs                       30 / 43 usable
  High Network Congestion (Sudden Spike)         32 / 37 usable

=== Fault window contexts (zone, app, mobility, congestion) ===
zone  app      mobility  congestion
A     Youtube  No        No            239
      File     No        No            159
C     File     No        No            136
      Youtube  No        No            125
B     Twitch   No        No             99

In [18]:
# =============================================================
# CELL 17 - Pure fault effects (within-window control) + statistical gating
# =============================================================
from scipy.stats import wilcoxon
import pickle

def window_signals(row):
    """Original CSV row -> (16, 128) array, using the verified extractor."""
    text = df_csv["KPIs"].values[row]
    return np.stack([extract_kpi_array(text, ch) for ch in NUMERICAL_CHANNELS])

EPS = 1e-6
MIN_OUTSIDE = 16
ALPHA = 0.001      # ~Bonferroni: 0.05 / (16 channels x 3 effect types)
LOG_CAP = 5.0      # caps ratios to x0.007 - x148 (floors prevent near-zero blow-ups)

# Level-type channels -> additive effects (native units, e.g. dB)
ADDITIVE = {"RSRP", "UL_SNR", "DL_MCS", "UL_MCS", "DL_BLER", "UL_BLER"}
# All other channels (traffic, PRB, buffer, packets) -> multiplicative effects

# --- Channel floors from 1,000 Normal windows (avoid dividing by ~0) ---
ref_rows = np.random.default_rng(SEED).choice(normal_rows, si   effects[fault] = {**rec, "affected": affected, "rows_used": np.array(used_rows)}
    print(f"\n{fault}  [{len(used_rows)} windows]")
    print(f"  affected ({affected.sum()}): " + ("; ".join(summary) if summary else "none"))

with open(os.path.join(SYN_DIR, "effects_v3.pkl"), "wb") as f:
    pickle.dump({"effects": effects, "LEVEL_FLOOR": LEVEL_FLOOR, "STD_FLOOR": STD_FLOOR,
                 "CH_MIN": CH_MIN, "CH_MAX": CH_MAX}, f)
print("\nSaved effects_v3.pkl")ze=1000, replace=False)
ref = np.stack([window_signals(r) for r in ref_rows])
LEVEL_FLOOR = 0.01 * np.median(np.abs(ref.mean(axis=2)), axis=0) + EPS
STD_FLOOR   = 0.01 * np.median(ref.std(axis=2), axis=0) + EPS

# --- Robust bounds (paper-style soft bounds) ---
# Normal: 0.1th / 99.9th percentile (removes glitches). Faults: full real range (keeps real extremes).
nm = train_y_str == NORMAL_LABEL
CH_MIN = np.array([min(np.percentile(train_X[nm, c * 4 + 2], 0.1), train_X[~nm, c * 4 + 2].min())
                   for c in range(16)])
CH_MAX = np.array([max(np.percentile(train_X[nm, c * 4 + 3], 99.9), train_X[~nm, c * 4 + 3].max())
                   for c in range(16)])

def fit_slope(t, y):
    return np.polyfit(t, y, 1)[0] if len(y) > 2 else 0.0

effects = {}
for fault in SYNTHETIC_FAULTS:
    rows = si_train[train_y_str == fault]
    rec = {k: [] for k in ["diff", "log_ratio", "log_std_ratio", "slope", "seg_std", "seg_mean"]}
    used_rows = []

    for r in rows:
        s, e = int(meta.loc[r, "anom_start"]), int(meta.loc[r, "anom_end"])
        if s < 0 or (128 - (e - s + 1)) < MIN_OUTSIDE:
            continue                                   # no normal part to compare with
        x = window_signals(r)                           # (16, 128)
        inside = np.zeros(128, bool); inside[s:e + 1] = True
        t_in, t_out = np.where(inside)[0], np.where(~inside)[0]

        diff = np.zeros(16)
        lr, lsr, slp = np.full(16, np.nan), np.full(16, np.nan), np.full(16, np.nan)
        for c, ch in enumerate(NUMERICAL_CHANNELS):
            seg, out = x[c, inside], x[c, ~inside]
            m_in, m_out = seg.mean(), out.mean()
            diff[c] = m_in - m_out                                          # additive level change
            if m_in > LEVEL_FLOOR[c] and m_out > LEVEL_FLOOR[c]:
                lr[c] = np.clip(np.log(m_in / m_out), -LOG_CAP, LOG_CAP)     # multiplicative level change
            if seg.std() > STD_FLOOR[c] and out.std() > STD_FLOOR[c]:
                lsr[c] = np.clip(np.log(seg.std() / out.std()), -LOG_CAP, LOG_CAP)   # variability change
            raw_slope = fit_slope(t_in, seg) - fit_slope(t_out, out)        # trend change
            if ch in ADDITIVE:
                slp[c] = raw_slope
            elif m_out > LEVEL_FLOOR[c]:
                slp[c] = np.clip(raw_slope / m_out, -LOG_CAP / 128, LOG_CAP / 128)

        rec["diff"].append(diff); rec["log_ratio"].append(lr)
        rec["log_std_ratio"].append(lsr); rec["slope"].append(slp)
        rec["seg_std"].append(x[ :, inside].std(axis=1)); rec["seg_mean"].append(x[:, inside].mean(axis=1))
        used_rows.append(r)

    rec = {k: np.array(v) for k, v in rec.items()}

    # --- Statistical gating: is the effect consistent across windows? ---
    affected, summary = np.zeros(16, bool), []
    for c, ch in enumerate(NUMERICAL_CHANNELS):
        tests = {"level": rec["diff"][:, c],            # always defined; Wilcoxon is rank-based
                 "var":   rec["log_std_ratio"][:, c],
                 "trend": rec["slope"][:, c]}
        hits = []
        for name, v in tests.items():
            v = v[np.isfinite(v)]
            if len(v) < 10 or np.allclose(v, 0):
                continue
            try:
                p = wilcoxon(v).pvalue
            except ValueError:
                continue
            if p < ALPHA:
                hits.append(f"{name}{'↑' if np.median(v) > 0 else '↓'}")
        if hits:
            affected[c] = True
            summary.append(f"{ch}({', '.join(hits)})")

    effects[fault] = {**rec, "affected": affected, "rows_used": np.array(used_rows)}
    print(f"\n{fault}  [{len(used_rows)} windows]")
    print(f"  affected ({affected.sum()}): " + ("; ".join(summary) if summary else "none"))

with open(os.path.join(SYN_DIR, "effects_v3.pkl"), "wb") as f:
    pickle.dump({"effects": effects, "LEVEL_FLOOR": LEVEL_FLOOR, "STD_FLOOR": STD_FLOOR,
                 "CH_MIN": CH_MIN, "CH_MAX": CH_MAX}, f)
print("\nSaved effects_v3.pkl")


Co-Channel Interference (Mild)  [53 windows]
  affected (6): RSRP(level↓); UL_BLER(level↓); UL_NPRB(level↑); UL_SNR(level↓); PRB_Utilization_DL(level↑); PRB_Utilization_UL(level↑)

Co-Channel Interference (Severe)  [44 windows]
  affected (8): RSRP(level↓, var↓); UL_BLER(level↑, var↑); UL_NPRB(level↑); UL_SNR(level↓); TX_Bytes(level↓, var↓); RX_Bytes(level↓, var↓); UL_NumberOfPackets(level↓, var↓); DL_NumberOfPackets(level↓, var↓)

Buffer Overflow (Gradual Buildup)  [61 windows]
  affected (10): DL_BLER(level↑); UL_BLER(level↑); UL_NPRB(level↑, trend↑); TX_Bytes(level↓); RX_Bytes(level↓, var↓); Estimated_UL_Buffer(level↑, var↑, trend↑); PRB_Utilization_DL(level↑, var↓, trend↑); PRB_Utilization_UL(level↑, var↑, trend↑); UL_NumberOfPackets(level↑, var↑, trend↑); DL_NumberOfPackets(level↑, trend↑)

Antenna Failure  [73 windows]
  affected (13): RSRP(level↓); DL_BLER(level↑); DL_MCS(level↓); UL_BLER(level↑); UL_MCS(level↓); UL_SNR(level↓); TX_Bytes(level↓); RX_Bytes(level↓); Estimated_UL_

In [19]:
# =============================================================
# CELL 17b - Clean host pool + full-window donors (context-referenced)
# =============================================================
# 1) Clean host pool: Normal windows in the fault state (Mobility=No, Congestion=No)
#    whose own values stay within the robust bounds.
row_to_tr = np.full(len(meta), -1)
row_to_tr[si_train] = np.arange(len(si_train))

in_state = (context["mobility"].values == "No") & (context["congestion"].values == "No")
cand = normal_rows[in_state[normal_rows]]
cfgA = train_X[row_to_tr[cand], :64]
clean = np.all((cfgA[:, 3::4] <= CH_MAX) & (cfgA[:, 2::4] >= CH_MIN), axis=1)
normal_pool = cand[clean]
print(f"Host pool: {len(normal_pool)} clean Normal windows (removed {(~clean).sum()} with out-of-bound values)")

# 2) Context-matched Normal reference: median window mean/std per (zone, app)
rng_ref = np.random.default_rng(SEED)
pool_zone = context["zone"].values[normal_pool]
pool_app  = context["app"].values[normal_pool]

def ref_stats(rows):
    pick = rng_ref.choice(rows, size=min(300, len(rows)), replace=False)
    xs = np.stack([window_signals(r) for r in pick])
    return np.median(xs.mean(axis=2), axis=0), np.median(xs.std(axis=2), axis=0)

REF = {}
for z, a in sorted(set(zip(pool_zone, pool_app))):
    REF[(z, a)] = ref_stats(normal_pool[(pool_zone == z) & (pool_app == a)])
REF_ZONE = {z: ref_stats(normal_pool[pool_zone == z]) for z in np.unique(pool_zone)}

# 3) Full-window donors: effect = fault segment vs context-matched Normal reference
for fault in SYNTHETIC_FAULTS:
    E = effects[fault]
    used = set(E["rows_used"].tolist())
    new = {k: [] for k in ["diff", "log_ratio", "log_std_ratio", "slope", "seg_std", "seg_mean"]}
    new_rows = []
    for r in si_train[train_y_str == fault]:
        if r in used:
            continue                                  # already a within-window donor
        s, e = int(meta.loc[r, "anom_start"]), int(meta.loc[r, "anom_end"])
        if s < 0 or e < s:
            s, e = 0, 127
        z, a = context.loc[r, "zone"], context.loc[r, "app"]
        ref_m, ref_sd = REF.get((z, a), REF_ZONE.get(z, (None, None)))
        if ref_m is None:
            continue
        x = window_signals(r)
        seg = x[:, s:e + 1]
        t_in = np.arange(e - s + 1)
        diff = np.zeros(16)
        lr, lsr, slp = np.full(16, np.nan), np.full(16, np.nan), np.full(16, np.nan)
        for c, ch in enumerate(NUMERICAL_CHANNELS):
            m_in, sd_in = seg[c].mean(), seg[c].std()
            diff[c] = m_in - ref_m[c]
            if m_in > LEVEL_FLOOR[c] and ref_m[c] > LEVEL_FLOOR[c]:
                lr[c] = np.clip(np.log(m_in / ref_m[c]), -LOG_CAP, LOG_CAP)
            if sd_in > STD_FLOOR[c] and ref_sd[c] > STD_FLOOR[c]:
                lsr[c] = np.clip(np.log(sd_in / ref_sd[c]), -LOG_CAP, LOG_CAP)
            raw_slope = fit_slope(t_in, seg[c])
            if ch in ADDITIVE:
                slp[c] = raw_slope
            elif ref_m[c] > LEVEL_FLOOR[c]:
                slp[c] = np.clip(raw_slope / ref_m[c], -LOG_CAP / 128, LOG_CAP / 128)
        new["diff"].append(diff); new["log_ratio"].append(lr)
        new["log_std_ratio"].append(lsr); new["slope"].append(slp)
        new["seg_std"].append(seg.std(axis=1)); new["seg_mean"].append(seg.mean(axis=1))
        new_rows.append(r)

    if new_rows:
        for k in new:
            E[k] = np.vstack([E[k], np.array(new[k])])
        E["rows_used"] = np.concatenate([E["rows_used"], np.array(new_rows)])
    print(f"{fault:<45s} donors: within-window={len(used):3d} + full-window={len(new_rows):3d} "
          f"= {len(E['rows_used'])}")

with open(os.path.join(SYN_DIR, "effects_v3.pkl"), "wb") as f:
    pickle.dump({"effects": effects, "LEVEL_FLOOR": LEVEL_FLOOR, "STD_FLOOR": STD_FLOOR,
                 "CH_MIN": CH_MIN, "CH_MAX": CH_MAX}, f)
print("\nUpdated effects_v3.pkl")

Host pool: 20910 clean Normal windows (removed 135 with out-of-bound values)
Co-Channel Interference (Mild)                donors: within-window= 53 + full-window= 73 = 126
Co-Channel Interference (Severe)              donors: within-window= 44 + full-window= 40 = 84
Buffer Overflow (Gradual Buildup)             donors: within-window= 61 + full-window= 56 = 117
Antenna Failure                               donors: within-window= 73 + full-window= 25 = 98
Faulty RF Filters (Temporal)                  donors: within-window= 41 + full-window= 26 = 67
High Network Congestion (Gradual Buildup)     donors: within-window= 29 + full-window= 27 = 56
Doppler Shift (Severe)                        donors: within-window= 48 + full-window= 29 = 77
Faulty Handover Algorithm (Too Frequent)      donors: within-window= 34 + full-window= 11 = 45
Resource Allocation Bugs                      donors: within-window= 30 + full-window= 13 = 43
High Network Congestion (Sudden Spike)        donors: within-windo

In [20]:
# =============================================================
# CELL 18 - Generator v3: context-controlled fault-effect transplantation
# =============================================================
rng = np.random.default_rng(SEED)
N_TARGET = 150

def array_to_config_A(x):
    """(16, 128) -> (64,): per channel [mean, std, min, max], same order as Config A."""
    return np.stack([x.mean(1), x.std(1), x.min(1), x.max(1)], axis=1).reshape(-1).astype(np.float32)



def pick_normal(zone, app):
    """Same zone + app if available; otherwise same zone (radio conditions)."""
    pool = normal_pool[(pool_zone == zone) & (pool_app == app)]
    fallback = len(pool) == 0
    if fallback:
        pool = normal_pool[pool_zone == zone]
    return pool[rng.integers(len(pool))], fallback

# Real windows per fault (for realistic segment positions)
all_rows = {f: si_train[train_y_str == f] for f in SYNTHETIC_FAULTS}
all_sess = {f: session_id_by_row[all_rows[f]] for f in SYNTHETIC_FAULTS}

def inject_v3(fault):
    E = effects[fault]
    k = rng.integers(len(E["rows_used"]))            # donor: one real fault window
    donor = E["rows_used"][k]
    donor_sess = session_id_by_row[donor]

    # Fault position: from a real window of the same fault AND same session as the donor
    same = all_rows[fault][all_sess[fault] == donor_sess]
    w = same[rng.integers(len(same))]
    s, e = int(meta.loc[w, "anom_start"]), int(meta.loc[w, "anom_end"])
    if s < 0 or e < s:
        s, e = 0, 127

    # Context-matched Normal window
    n_row, fb = pick_normal(context.loc[donor, "zone"], context.loc[donor, "app"])
    x = window_signals(n_row).astype(np.float64)
    tc = np.arange(e - s + 1) - (e - s) / 2.0        # centered time axis for the trend

    for c, ch in enumerate(NUMERICAL_CHANNELS):
        if not E["affected"][c]:
            continue
        seg = x[c, s:e + 1]
        m, sd = seg.mean(), seg.std()
        lr, lsr, slp = E["log_ratio"][k, c], E["log_std_ratio"][k, c], E["slope"][k, c]

        # 1) Level
        if ch not in ADDITIVE and np.isfinite(lr) and m > LEVEL_FLOOR[c]:
            new_m = m * np.exp(lr)                   # multiplicative (traffic-type)
        else:
            new_m = m + E["diff"][k, c]              # additive (dB, MCS, BLER) or fallback

        # 2) Variability
        if sd > STD_FLOOR[c] and np.isfinite(lsr):
            dev = (seg - m) * np.exp(lsr)
        elif sd <= STD_FLOOR[c]:                     # flat signal: create variability
            d_sd = E["seg_std"][k, c]
            if ch not in ADDITIVE and E["seg_mean"][k, c] > LEVEL_FLOOR[c]:
                d_sd = d_sd / E["seg_mean"][k, c] * max(abs(new_m), LEVEL_FLOOR[c])
            dev = rng.normal(0, d_sd, len(seg))
        else:
            dev = seg - m

        # 3) Trend
        trend = 0.0
        if np.isfinite(slp):
            scale = 1.0 if ch in ADDITIVE else max(abs(m), LEVEL_FLOOR[c])
            trend = slp * scale * tc

        # 4) Bounds: physical limits, then the range real data actually reaches
        seg_new = clip_channel(new_m + dev + trend, ch)
        x[c, s:e + 1] = np.clip(seg_new, CH_MIN[c], CH_MAX[c])
    return x, n_row, donor_sess, fb

# --- Generate up to N_TARGET per class ---
v3_X, v3_y, v3_src_sess, v3_donor_sess = [], [], [], []
for fault in SYNTHETIC_FAULTS:
    needed = max(0, N_TARGET - len(all_rows[fault]))
    n_fb = 0
    for _ in range(needed):
        x, n_row, d_sess, fb = inject_v3(fault)
        v3_X.append(array_to_config_A(x))
        v3_y.append(fault)
        v3_src_sess.append(session_id_by_row[n_row])
        v3_donor_sess.append(d_sess)
        n_fb += fb
    print(f"{fault:<45s} generated={needed:3d}  context-fallbacks={n_fb}")
# Jamming already has 185 real windows (> 150): none generated.

v3_X = np.array(v3_X, dtype=np.float32)
v3_y = np.array(v3_y)
v3_src_sess = np.array(v3_src_sess)
v3_donor_sess = np.array(v3_donor_sess)

np.save(os.path.join(SYN_DIR, "syn_v3_features.npy"), v3_X)
np.save(os.path.join(SYN_DIR, "syn_v3_labels.npy"), v3_y)
np.save(os.path.join(SYN_DIR, "syn_v3_src_sessions.npy"), v3_src_sess)
np.save(os.path.join(SYN_DIR, "syn_v3_donor_sessions.npy"), v3_donor_sess)

# --- Diagnostic: synthetic vs real range per channel (including extremes) ---
real_fault_X = train_X[train_y_str != NORMAL_LABEL][:, :64]
print(f"\nv3 overall: mean={v3_X.mean():.1f}, std={v3_X.std():.1f}  "
      f"(real: mean={real_fault_X.mean():.1f}, std={real_fault_X.std():.1f})")
print(f"\n{'Channel':<22}{'real p99':>12}{'syn p99':>12}{'real max':>12}{'syn max':>12}")
for c, ch in enumerate(NUMERICAL_CHANNELS):
    col = c * 4 + 3
    print(f"{ch:<22}"
          f"{np.percentile(real_fault_X[:, col], 99):>12.1f}{np.percentile(v3_X[:, col], 99):>12.1f}"
          f"{real_fault_X[:, col].max():>12.1f}{v3_X[:, col].max():>12.1f}")

Co-Channel Interference (Mild)                generated= 24  context-fallbacks=10
Co-Channel Interference (Severe)              generated= 66  context-fallbacks=7
Buffer Overflow (Gradual Buildup)             generated= 33  context-fallbacks=5
Antenna Failure                               generated= 52  context-fallbacks=8
Faulty RF Filters (Temporal)                  generated= 83  context-fallbacks=0
High Network Congestion (Gradual Buildup)     generated= 94  context-fallbacks=39
Doppler Shift (Severe)                        generated= 73  context-fallbacks=0
Faulty Handover Algorithm (Too Frequent)      generated=105  context-fallbacks=0
Resource Allocation Bugs                      generated=107  context-fallbacks=0
High Network Congestion (Sudden Spike)        generated=113  context-fallbacks=12

v3 overall: mean=9039.6, std=55658.6  (real: mean=7931.7, std=49961.3)

Channel                   real p99     syn p99    real max     syn max
RSRP                         -73.0       -7

In [38]:
# =============================================================
# CELL 19 - Synthetic quality: TSTR + class-matched realism
# =============================================================
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier

# Real windows of the 10 synthetic fault classes
m10 = np.isin(train_y_str, SYNTHETIC_FAULTS)
rX, ry = train_X[m10][:, :64], train_y_str[m10]
rs = session_id_by_row[si_train][m10]

versions = {
    "v1": (np.load(os.path.join(SYN_DIR, "syn_features.npy")),
           np.load(os.path.join(SYN_DIR, "syn_labels.npy")),
           np.load(os.path.join(SYN_DIR, "syn_sessions.npy")),
           np.load(os.path.join(SYN_DIR, "syn_sessions.npy"))),        # v1 has no donors
    "v3": (v3_X, v3_y, v3_src_sess, v3_donor_sess),
}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(cv.split(rX, ry, groups=rs))

def tstr(sX, sy, s_src, s_don):
    """Train on synthetic only, test on real windows of unseen sessions."""
    f1s = []
    for _, te in folds:
        test_sess = set(np.unique(rs[te]))
        keep = np.array([(a not in test_sess) and (b not in test_sess) for a, b in zip(s_src, s_don)])
        clf = CatBoostClassifier(random_seed=SEED, verbose=0, thread_count=4)
        clf.fit(sX[keep], sy[keep])
        f1s.append(f1_score(ry[te], clf.predict(rX[te]).ravel(), average="macro", zero_division=0))
    return np.mean(f1s), np.std(f1s)

def trtr():
    """Reference: train on real, test on real (same folds)."""
    f1s = []
    for tr, te in folds:
        clf = CatBoostClassifier(random_seed=SEED, verbose=0, thread_count=4)
        clf.fit(rX[tr], ry[tr])
        f1s.append(f1_score(ry[te], clf.predict(rX[te]).ravel(), average="macro", zero_division=0))
    return np.mean(f1s), np.std(f1s)

print("=== TSTR: Train on Synthetic, Test on Real (10 fault classes) ===")
print("Chance level ≈ 0.10 | higher = synthetic captured real fault signatures\n")
m, s = trtr()
print(f"  TRTR (real -> real, reference): {m:.4f} ± {s:.4f}")
for name, (sX, sy, s_src, s_don) in versions.items():
    m, s = tstr(sX, sy, s_src, s_don)
    print(f"  TSTR {name} (synthetic -> real):  {m:.4f} ± {s:.4f}")

print("\n=== Class-matched realism AUC (lower = more realistic, 0.5 = indistinguishable) ===")
rng_chk = np.random.default_rng(SEED)
for name, (sX, sy, _, _) in versions.items():
    Xr, Xs = [], []
    for fault in np.unique(sy):
        r, sy_f = rX[ry == fault], sX[sy == fault]
        n = min(len(r), len(sy_f))
        if n < 5:
            continue
        Xr.append(r[rng_chk.choice(len(r), n, replace=False)])
        Xs.append(sy_f[rng_chk.choice(len(sy_f), n, replace=False)])
    Xc = np.vstack(Xr + Xs)
    yc = np.array([0] * sum(len(a) for a in Xr) + [1] * sum(len(a) for a in Xs))
    auc = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
                          Xc, yc, cv=5, scoring="roc_auc")
    print(f"  {name}: {auc.mean():.3f} ± {auc.std():.3f}")

=== TSTR: Train on Synthetic, Test on Real (10 fault classes) ===
Chance level ≈ 0.10 | higher = synthetic captured real fault signatures

  TRTR (real -> real, reference): 0.3108 ± 0.0894
  TSTR v1 (synthetic -> real):  0.1195 ± 0.0503
  TSTR v3 (synthetic -> real):  0.3931 ± 0.1191

=== Class-matched realism AUC (lower = more realistic, 0.5 = indistinguishable) ===
  v1: 0.994 ± 0.005
  v3: 0.938 ± 0.035


In [39]:
# =============================================================
# CELL 20 - CV experiment: Real only vs Real + Synthetic v3 (11 classes)
# =============================================================
from collections import Counter

fault_mask = train_y_str != NORMAL_LABEL
real_X, real_y = train_X[fault_mask][:, :64], train_y_str[fault_mask]
real_sessions = session_id_by_row[si_train][fault_mask]

def run_cv_v3(use_syn, label, k=5):
    cv = StratifiedGroupKFold(n_splits=min(k, len(np.unique(real_sessions))),
                              shuffle=True, random_state=SEED)
    f1s = []
    for i, (tr, va) in enumerate(cv.split(real_X, real_y, groups=real_sessions)):
        X_tr, y_tr = real_X[tr], real_y[tr]
        if use_syn:
            val_sess = set(np.unique(real_sessions[va]))
            keep = np.array([(a not in val_sess) and (b not in val_sess)
                             for a, b in zip(v3_src_sess, v3_donor_sess)])
            X_tr = np.vstack([X_tr, v3_X[keep]])
            y_tr = np.concatenate([y_tr, v3_y[keep]])
        counts = Counter(y_tr)
        w = {c: len(y_tr) / (len(counts) * n) for c, n in counts.items()}
        clf = CatBoostClassifier(random_seed=SEED, verbose=0, thread_count=4, class_weights=w)
        clf.fit(X_tr, y_tr)
        f1 = f1_score(real_y[va], clf.predict(real_X[va]).ravel(), average="macro", zero_division=0)
        f1s.append(f1)
        print(f"  Fold {i+1}: {f1:.4f}")
    m, s = np.mean(f1s), np.std(f1s)
    print(f"{label}: {m:.4f} ± {s:.4f}\n")
    return m, s

print("=== Real + Synthetic v3 ===")
v3_mean, v3_std = run_cv_v3(True, "Real + Synthetic v3")

with open(os.path.join(SYN_DIR, "synthetic_experiment_results.json")) as f:
    prev = json.load(f)

print("=" * 55)
print(f"{'Real only':<28}{prev['baseline_mean']:.4f} ± {prev['baseline_std']:.4f}")
print(f"{'Real + Synthetic v1':<28}{prev['augmented_mean']:.4f} ± {prev['augmented_std']:.4f}")
print(f"{'Real + Synthetic v3':<28}{v3_mean:.4f} ± {v3_std:.4f}")
print(f"{'Delta v3 vs real only':<28}{v3_mean - prev['baseline_mean']:+.4f}")
print("=" * 55)

with open(os.path.join(SYN_DIR, "synthetic_v3_results.json"), "w") as f:
    json.dump({"v3_mean": float(v3_mean), "v3_std": float(v3_std),
               "baseline_mean": prev["baseline_mean"],
               "delta_vs_baseline": float(v3_mean - prev["baseline_mean"])}, f, indent=2)
print("Saved synthetic_v3_results.json")

=== Real + Synthetic v3 ===
  Fold 1: 0.2761
  Fold 2: 0.3478
  Fold 3: 0.4322
  Fold 4: 0.2477
  Fold 5: 0.4295
Real + Synthetic v3: 0.3467 ± 0.0761

Real only                   0.2942 ± 0.0819
Real + Synthetic v1         0.2863 ± 0.1031
Real + Synthetic v3         0.3467 ± 0.0761
Delta v3 vs real only       +0.0524
Saved synthetic_v3_results.json


In [21]:
# =============================================================
# CELL 21 - Signal cache for nested CV (parse once, reuse)
# =============================================================
CACHE_PATH = os.path.join(SYN_DIR, "signal_cache_v3.npz")
rng_c = np.random.default_rng(SEED)

# Real training windows of the 10 synthetic fault classes
fault10_rows = si_train[np.isin(train_y_str, SYNTHETIC_FAULTS)]

# Host subset: up to 600 clean Normal windows per (zone, app) - plenty for generation
hz = context["zone"].values[normal_pool]
ha = context["app"].values[normal_pool]
host_rows = []
for z, a in sorted(set(zip(hz, ha))):
    grp = normal_pool[(hz == z) & (ha == a)]
    host_rows.extend(rng_c.choice(grp, size=min(600, len(grp)), replace=False))
host_rows = np.array(sorted(host_rows))

cache_rows = np.concatenate([fault10_rows, host_rows])
if os.path.exists(CACHE_PATH) and np.array_equal(np.load(CACHE_PATH)["rows"], cache_rows):
    cache_sig = np.load(CACHE_PATH)["signals"]
    print("Loaded signal cache.")
else:
    cache_sig = np.zeros((len(cache_rows), 16, 128), dtype=np.float64)
    for i, r in enumerate(cache_rows):
        cache_sig[i] = window_signals(r)
        if i % 1000 == 0:
            print(f"  cached {i}/{len(cache_rows)}")
    np.savez(CACHE_PATH, rows=cache_rows, signals=cache_sig)
    print("Signal cache saved.")

ROW2IDX = {int(r): i for i, r in enumerate(cache_rows)}
def sig(r):
    return cache_sig[ROW2IDX[int(r)]]

print(f"Fault windows cached: {len(fault10_rows)} | host windows cached: {len(host_rows)}")

  cached 0/4950
  cached 1000/4950
  cached 2000/4950
  cached 3000/4950
  cached 4000/4950
Signal cache saved.
Fault windows cached: 750 | host windows cached: 4200


In [26]:
# =============================================================
# CELL 22a - Nested, repeated CV (5 seeds x 5 folds) + paired statistics
# =============================================================
# Nested: in EACH fold, effects, channel gating, references, bounds and the
# synthetic data are rebuilt from TRAINING-fold sessions only.
from scipy.stats import wilcoxon, t as t_dist
from collections import Counter

SEEDS = [42, 7, 123, 2024, 99]
KEYS = ["diff", "log_ratio", "log_std_ratio", "slope", "seg_std", "seg_mean"]
LABELS = meta["label"].values

fault_mask = train_y_str != NORMAL_LABEL
real_rows = si_train[fault_mask]
real_X, real_y = train_X[fault_mask][:, :64], train_y_str[fault_mask]
real_sess = session_id_by_row[real_rows]
host_sess = session_id_by_row[host_rows]

nm = train_y_str == NORMAL_LABEL
NORMAL_MIN = np.array([np.percentile(train_X[nm, c * 4 + 2], 0.1) for c in range(16)])
NORMAL_MAX = np.array([np.percentile(train_X[nm, c * 4 + 3], 99.9) for c in range(16)])

def seg_bounds(r):
    s, e = int(meta.loc[r, "anom_start"]), int(meta.loc[r, "anom_end"])
    return (0, 127) if (s < 0 or e < s) else (s, e)

def compute_effect(x, s, e, ref=None):
    """Effect of the fault segment vs the normal part of the window (ref=None)
    or vs a context-matched Normal reference (ref=(mean16, std16))."""
    inside = np.zeros(128, bool); inside[s:e + 1] = True
    t_in, t_out = np.where(inside)[0], np.where(~inside)[0]
    diff = np.zeros(16)
    lr, lsr, slp = np.full(16, np.nan), np.full(16, np.nan), np.full(16, np.nan)
    for c, ch in enumerate(NUMERICAL_CHANNELS):
        seg = x[c, inside]
        if ref is None:
            out = x[c, ~inside]
            m_out, sd_out = out.mean(), out.std()
            raw = fit_slope(t_in, seg) - fit_slope(t_out, out)
        else:
            m_out, sd_out = ref[0][c], ref[1][c]
            raw = fit_slope(t_in, seg)
        m_in, sd_in = seg.mean(), seg.std()
        diff[c] = m_in - m_out
        if m_in > LEVEL_FLOOR[c] and m_out > LEVEL_FLOOR[c]:
            lr[c] = np.clip(np.log(m_in / m_out), -LOG_CAP, LOG_CAP)
        if sd_in > STD_FLOOR[c] and sd_out > STD_FLOOR[c]:
            lsr[c] = np.clip(np.log(sd_in / sd_out), -LOG_CAP, LOG_CAP)
        if ch in ADDITIVE:
            slp[c] = raw
        elif m_out > LEVEL_FLOOR[c]:
            slp[c] = np.clip(raw / m_out, -LOG_CAP / 128, LOG_CAP / 128)
    return diff, lr, lsr, slp, x[:, inside].std(1), x[:, inside].mean(1)

def estimate_effects(train_rows, hosts):
    hz, ha = context["zone"].values[hosts], context["app"].values[hosts]
    ref_of = lambda rows: (np.median(np.stack([sig(r) for r in rows]).mean(2), 0),
                           np.median(np.stack([sig(r) for r in rows]).std(2), 0))
    REF_f = {(z, a): ref_of(hosts[(hz == z) & (ha == a)]) for z, a in sorted(set(zip(hz, ha)))}
    REFZ_f = {z: ref_of(hosts[hz == z]) for z in sorted(set(hz))}

    eff = {}
    for fault in SYNTHETIC_FAULTS:
        rows = train_rows[LABELS[train_rows] == fault]
        rec, donors, within_idx = {k: [] for k in KEYS}, [], []
        for r in rows:
            s, e = seg_bounds(r)
            if (128 - (e - s + 1)) >= MIN_OUTSIDE:
                vals = compute_effect(sig(r), s, e)                     # within-window donor
                within_idx.append(len(donors))
            else:
                z, a = context.loc[r, "zone"], context.loc[r, "app"]
                ref = REF_f.get((z, a), REFZ_f.get(z))
                if ref is None:
                    continue
                vals = compute_effect(sig(r), s, e, ref)                # full-window donor
            for k, v in zip(KEYS, vals):
                rec[k].append(v)
            donors.append(r)
        rec = {k: np.array(v) for k, v in rec.items()}

        # Gating on within-window donors only (context-free evidence)
        wi, affected = np.array(within_idx, int), np.zeros(16, bool)
        for c in range(16):
            for arr in (rec["diff"][wi, c], rec["log_std_ratio"][wi, c], rec["slope"][wi, c]):
                v = arr[np.isfinite(arr)]
                if len(v) < 10 or np.allclose(v, 0):
                    continue
                try:
                    if wilcoxon(v).pvalue < ALPHA:
                        affected[c] = True
                        break
                except ValueError:
                    pass
        eff[fault] = {**rec, "affected": affected, "rows_used": np.array(donors)}
    return eff

def generate(eff, train_rows, hosts, rng, ch_min, ch_max):
    hz, ha = context["zone"].values[hosts], context["app"].values[hosts]
    X, Y = [], []
    for fault in SYNTHETIC_FAULTS:
        E = eff[fault]
        if E["affected"].sum() == 0 or len(E["rows_used"]) == 0:
            print(f"    skip {fault}: no reliable effect in this fold")
            continue
        f_rows = train_rows[LABELS[train_rows] == fault]
        f_sess = session_id_by_row[f_rows]
        for _ in range(max(0, N_TARGET - len(f_rows))):
            k = rng.integers(len(E["rows_used"]))
            donor = E["rows_used"][k]
            same = f_rows[f_sess == session_id_by_row[donor]]
            s, e = seg_bounds(same[rng.integers(len(same))])
            z, a = context.loc[donor, "zone"], context.loc[donor, "app"]
            pool = hosts[(hz == z) & (ha == a)]
            if len(pool) == 0:
                pool = hosts[hz == z]
            if len(pool) == 0:
                pool = hosts
            x = sig(pool[rng.integers(len(pool))]).copy()
            tc = np.arange(e - s + 1) - (e - s) / 2.0
            for c, ch in enumerate(NUMERICAL_CHANNELS):
                if not E["affected"][c]:
                    continue
                seg = x[c, s:e + 1]
                m, sd = seg.mean(), seg.std()
                lr, lsr, slp = E["log_ratio"][k, c], E["log_std_ratio"][k, c], E["slope"][k, c]
                new_m = m * np.exp(lr) if (ch not in ADDITIVE and np.isfinite(lr) and m > LEVEL_FLOOR[c]) \
                        else m + E["diff"][k, c]
                if sd > STD_FLOOR[c] and np.isfinite(lsr):
                    dev = (seg - m) * np.exp(lsr)
                elif sd <= STD_FLOOR[c]:
                    d_sd = E["seg_std"][k, c]
                    if ch not in ADDITIVE and E["seg_mean"][k, c] > LEVEL_FLOOR[c]:
                        d_sd = d_sd / E["seg_mean"][k, c] * max(abs(new_m), LEVEL_FLOOR[c])
                    dev = rng.normal(0, d_sd, len(seg))
                else:
                    dev = seg - m
                trend = 0.0
                if np.isfinite(slp):
                    trend = slp * (1.0 if ch in ADDITIVE else max(abs(m), LEVEL_FLOOR[c])) * tc
                x[c, s:e + 1] = np.clip(clip_channel(new_m + dev + trend, ch), ch_min[c], ch_max[c])
            X.append(array_to_config_A(x)); Y.append(fault)
    return np.array(X, dtype=np.float32), np.array(Y)

def fit_eval(X_tr, y_tr, X_va, y_va, seed):
    counts = Counter(y_tr)
    w = {c: len(y_tr) / (len(counts) * n) for c, n in counts.items()}
    clf = CatBoostClassifier(random_seed=seed, verbose=0, thread_count=4, class_weights=w)
    clf.fit(X_tr, y_tr)
    return f1_score(y_va, clf.predict(X_va).ravel(), average="macro", zero_division=0)


In [25]:

# ---------------- Main loop ----------------
records = []
for seed in SEEDS:
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for f_i, (tr, va) in enumerate(cv.split(real_X, real_y, groups=real_sess)):
        val_sess = set(real_sess[va].tolist())
        tr_rows = real_rows[tr]
        hosts_ok = host_rows[~np.isin(host_sess, list(val_sess))]

        # Fold-specific bounds (training-fold faults + Normal)
        ch_min = np.minimum(NORMAL_MIN, real_X[tr][:, 2::4].min(0))
        ch_max = np.maximum(NORMAL_MAX, real_X[tr][:, 3::4].max(0))

        eff = estimate_effects(tr_rows, hosts_ok)
        synX, synY = generate(eff, tr_rows, hosts_ok, np.random.default_rng(seed * 100 + f_i),
                              ch_min, ch_max)

        f_base = fit_eval(real_X[tr], real_y[tr], real_X[va], real_y[va], seed)
        f_aug = fit_eval(np.vstack([real_X[tr], synX]), np.concatenate([real_y[tr], synY]),
                         real_X[va], real_y[va], seed)
        records.append({"seed": seed, "fold": f_i + 1, "base": f_base, "aug": f_aug,
                        "n_train": int(len(tr)), "n_val": int(len(va)), "n_syn": int(len(synY))})
        print(f"seed={seed:<5} fold={f_i+1}  real-only={f_base:.4f}  real+v3={f_aug:.4f}  "
              f"Δ={f_aug - f_base:+.4f}  (synthetic={len(synY)})")

# ---------------- Paired statistics ----------------
res = pd.DataFrame(records)
d = (res["aug"] - res["base"]).values
n = len(d)

w_p = wilcoxon(d).pvalue
ratio = np.mean(res["n_val"] / res["n_train"])
t_stat = d.mean() / np.sqrt((1 / n + ratio) * d.var(ddof=1))     # Nadeau-Bengio corrected t-test
t_p = 2 * (1 - t_dist.cdf(abs(t_stat), df=n - 1))

print("\n" + "=" * 62)
print(f"Real only          : {res['base'].mean():.4f} ± {res['base'].std():.4f}")
print(f"Real + v3 (nested) : {res['aug'].mean():.4f} ± {res['aug'].std():.4f}")
print(f"Mean paired Δ      : {d.mean():+.4f}   (wins {int((d > 0).sum())}/{n})")
print(f"Wilcoxon p         : {w_p:.4f}")
print(f"Corrected t-test p : {t_p:.4f}   (Nadeau-Bengio, accounts for overlapping CV folds)")
print("=" * 62)
print("\nPer-seed means:")
print(res.groupby("seed")[["base", "aug"]].mean().round(4).to_string())

res.to_csv(os.path.join(SYN_DIR, "nested_cv_records.csv"), index=False)
with open(os.path.join(SYN_DIR, "nested_cv_summary.json"), "w") as f:
    json.dump({"base_mean": float(res["base"].mean()), "aug_mean": float(res["aug"].mean()),
               "mean_delta": float(d.mean()), "wins": int((d > 0).sum()), "n": n,
               "wilcoxon_p": float(w_p), "corrected_t_p": float(t_p)}, f, indent=2)
print("\nSaved nested_cv_records.csv and nested_cv_summary.json")

In [27]:
# =============================================================
# CELL 23 - Screening sweep: synthetic amount x synthetic weight
#           (nested CV, 2 seeds; checkpointed after every fold)
# =============================================================
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier
from collections import Counter
import time

AMOUNTS     = [100, 150, 300, 500]   # target windows per class (real + synthetic)
SYN_WEIGHTS = [1.0, 0.5]             # how much each synthetic window counts vs a real one
SWEEP_SEEDS = [42, 7]
N_THREADS   = max(1, (os.cpu_count() or 4) - 2)   # leave 2 cores free
CKPT = os.path.join(SYN_DIR, "sweep_records_partial.csv")

def fit_eval_w(X_tr, y_tr, w_tr, X_va, y_va, seed):
    """Class weights from effective (weighted) counts x per-sample synthetic weight."""
    classes = np.unique(y_tr)
    eff = {c: w_tr[y_tr == c].sum() for c in classes}
    total = sum(eff.values())
    cw = {c: total / (len(classes) * eff[c]) for c in classes}
    sw = w_tr * np.array([cw[c] for c in y_tr])
    clf = CatBoostClassifier(random_seed=seed, verbose=0, thread_count=N_THREADS)
    clf.fit(X_tr, y_tr, sample_weight=sw)
    return f1_score(y_va, clf.predict(X_va).ravel(), average="macro", zero_division=0)

# --- Resume from checkpoint if it exists ---
rows_out, done = [], set()
if os.path.exists(CKPT):
    prev = pd.read_csv(CKPT)
    rows_out = prev.to_dict("records")
    done = set(zip(prev["seed"], prev["fold"]))
    print(f"Resuming: {len(done)} fold(s) already done -> skipped")

t0 = time.time()
for seed in SWEEP_SEEDS:
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for f_i, (tr, va) in enumerate(cv.split(real_X, real_y, groups=real_sess)):
        if (seed, f_i + 1) in done:
            continue
        val_sess = set(real_sess[va].tolist())
        tr_rows  = real_rows[tr]
        hosts_ok = host_rows[~np.isin(host_sess, list(val_sess))]
        ch_min = np.minimum(NORMAL_MIN, real_X[tr][:, 2::4].min(0))
        ch_max = np.maximum(NORMAL_MAX, real_X[tr][:, 3::4].max(0))

        eff = estimate_effects(tr_rows, hosts_ok)

        N_TARGET = max(AMOUNTS)   # generate the largest set once; smaller = first part of it
        synX_all, synY_all = generate(eff, tr_rows, hosts_ok,
                                      np.random.default_rng(seed * 100 + f_i), ch_min, ch_max)

        Xtr, ytr = real_X[tr], real_y[tr]
        real_counts = Counter(ytr)
        fold_rows = []

        base = fit_eval_w(Xtr, ytr, np.ones(len(ytr)), real_X[va], real_y[va], seed)
        fold_rows.append({"seed": seed, "fold": f_i + 1, "amount": 0, "weight": 0.0, "f1": base})

        for amt in AMOUNTS:
            keep = []
            for f in SYNTHETIC_FAULTS:
                idx = np.where(synY_all == f)[0]
                keep.extend(idx[:max(0, amt - real_counts.get(f, 0))])
            keep = np.array(keep, dtype=int)
            for w_syn in SYN_WEIGHTS:
                X = np.vstack([Xtr, synX_all[keep]])
                y = np.concatenate([ytr, synY_all[keep]])
                w = np.concatenate([np.ones(len(ytr)), np.full(len(keep), w_syn)])
                f1 = fit_eval_w(X, y, w, real_X[va], real_y[va], seed)
                fold_rows.append({"seed": seed, "fold": f_i + 1, "amount": amt,
                                  "weight": w_syn, "f1": f1})

        rows_out.extend(fold_rows)
        pd.DataFrame(rows_out).to_csv(CKPT, index=False)    # checkpoint after every fold
        print(f"seed={seed} fold={f_i+1} done & saved  |  baseline={base:.4f}  |  "
              f"elapsed {(time.time() - t0) / 60:.1f} min")

N_TARGET = 150   # restore the default

# ---------------- Summary ----------------
sw = pd.DataFrame(rows_out)
base_df = sw[sw.amount == 0][["seed", "fold", "f1"]].rename(columns={"f1": "base"})
res = sw[sw.amount > 0].merge(base_df, on=["seed", "fold"])
res["delta"] = res["f1"] - res["base"]

summary = (res.groupby(["amount", "weight"])
              .agg(mean_f1=("f1", "mean"), mean_delta=("delta", "mean"),
                   wins=("delta", lambda d: int((d > 0).sum())), n=("delta", "size"))
              .reset_index().sort_values("mean_delta", ascending=False))

print(f"\nBaseline (real only): {base_df['base'].mean():.4f}\n")
print(summary.round(4).to_string(index=False))

sw.to_csv(os.path.join(SYN_DIR, "sweep_records.csv"), index=False)
summary.to_csv(os.path.join(SYN_DIR, "sweep_summary.csv"), index=False)
print("\nSaved sweep_records.csv and sweep_summary.csv")

Resuming: 4 fold(s) already done -> skipped
seed=42 fold=5 done & saved  |  baseline=0.4491  |  elapsed 2.6 min
seed=7 fold=1 done & saved  |  baseline=0.2350  |  elapsed 5.2 min
seed=7 fold=2 done & saved  |  baseline=0.2856  |  elapsed 7.7 min
    skip High Network Congestion (Gradual Buildup): no reliable effect in this fold
seed=7 fold=3 done & saved  |  baseline=0.2932  |  elapsed 10.1 min
seed=7 fold=4 done & saved  |  baseline=0.2143  |  elapsed 12.7 min
seed=7 fold=5 done & saved  |  baseline=0.4651  |  elapsed 15.1 min

Baseline (real only): 0.2964

 amount  weight  mean_f1  mean_delta  wins  n
    500     1.0   0.3513      0.0549     9 10
    300     0.5   0.3436      0.0471     7 10
    500     0.5   0.3436      0.0471     8 10
    300     1.0   0.3407      0.0443     7 10
    150     1.0   0.3375      0.0411     8 10
    100     1.0   0.3292      0.0328     9 10
    150     0.5   0.3255      0.0290     7 10
    100     0.5   0.3210      0.0245     8 10

Saved sweep_records.

In [28]:
# =============================================================
# CELL 24 - Per-class breakdown: real only vs real + v3 (winner setting)
# =============================================================
from sklearn.metrics import precision_recall_fscore_support

BEST_AMOUNT, BEST_WEIGHT = 500, 1.0
BD_SEEDS = [42, 7]

def fit_predict_w(X_tr, y_tr, w_tr, X_va, seed):
    classes_ = np.unique(y_tr)
    eff_ = {c: w_tr[y_tr == c].sum() for c in classes_}
    total = sum(eff_.values())
    cw = {c: total / (len(classes_) * eff_[c]) for c in classes_}
    sw = w_tr * np.array([cw[c] for c in y_tr])
    clf = CatBoostClassifier(random_seed=seed, verbose=0, thread_count=N_THREADS)
    clf.fit(X_tr, y_tr, sample_weight=sw)
    return clf.predict(X_va).ravel()

classes = np.unique(real_y)
oof = {(m, s): np.empty(len(real_y), dtype=object) for m in ("base", "v3") for s in BD_SEEDS}

N_TARGET = BEST_AMOUNT
for seed in BD_SEEDS:
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for f_i, (tr, va) in enumerate(cv.split(real_X, real_y, groups=real_sess)):
        val_sess = set(real_sess[va].tolist())
        tr_rows  = real_rows[tr]
        hosts_ok = host_rows[~np.isin(host_sess, list(val_sess))]
        ch_min = np.minimum(NORMAL_MIN, real_X[tr][:, 2::4].min(0))
        ch_max = np.maximum(NORMAL_MAX, real_X[tr][:, 3::4].max(0))

        eff = estimate_effects(tr_rows, hosts_ok)
        synX, synY = generate(eff, tr_rows, hosts_ok,
                              np.random.default_rng(seed * 100 + f_i), ch_min, ch_max)

        Xtr, ytr = real_X[tr], real_y[tr]
        oof[("base", seed)][va] = fit_predict_w(Xtr, ytr, np.ones(len(ytr)), real_X[va], seed)
        X = np.vstack([Xtr, synX])
        y = np.concatenate([ytr, synY])
        w = np.concatenate([np.ones(len(ytr)), np.full(len(synY), BEST_WEIGHT)])
        oof[("v3", seed)][va] = fit_predict_w(X, y, w, real_X[va], seed)
        print(f"seed={seed} fold={f_i+1} done")
N_TARGET = 150

# --- Per-class F1 (out-of-fold predictions, averaged over seeds) ---
def f1_per_class(pred):
    _, _, f1, sup = precision_recall_fscore_support(real_y, pred, labels=classes, zero_division=0)
    return f1, sup

f1_base = np.mean([f1_per_class(oof[("base", s)])[0] for s in BD_SEEDS], axis=0)
f1_v3   = np.mean([f1_per_class(oof[("v3", s)])[0] for s in BD_SEEDS], axis=0)
support = f1_per_class(oof[("base", BD_SEEDS[0])])[1]

tab = pd.DataFrame({"class": classes, "support": support,
                    "F1_real_only": f1_base, "F1_real_v3": f1_v3})
tab["delta"] = tab["F1_real_v3"] - tab["F1_real_only"]
tab = tab.sort_values("F1_real_v3")
print("\n=== Per-class F1 (weakest first) ===")
print(tab.round(3).to_string(index=False))
print(f"\nMacro F1 (pooled): real only = {f1_base.mean():.4f} | real + v3 = {f1_v3.mean():.4f}")

# --- Top confusions with v3 (seed 42) ---
cm = pd.crosstab(pd.Series(real_y, name="true"), pd.Series(oof[("v3", 42)], name="pred"))
pairs = [(t, p, int(cm.loc[t, p]), cm.loc[t, p] / cm.loc[t].sum())
         for t in cm.index for p in cm.columns if t != p and cm.loc[t, p] > 0]
conf = (pd.DataFrame(pairs, columns=["true", "predicted_as", "count", "share_of_true"])
          .sort_values("count", ascending=False).head(10))
print("\n=== Top 10 confusions (real + v3) ===")
print(conf.round(2).to_string(index=False))

tab.to_csv(os.path.join(SYN_DIR, "per_class_breakdown.csv"), index=False)
conf.to_csv(os.path.join(SYN_DIR, "top_confusions_v3.csv"), index=False)
print("\nSaved per_class_breakdown.csv and top_confusions_v3.csv")

seed=42 fold=1 done
seed=42 fold=2 done
    skip High Network Congestion (Gradual Buildup): no reliable effect in this fold
seed=42 fold=3 done
seed=42 fold=4 done
seed=42 fold=5 done
seed=7 fold=1 done
seed=7 fold=2 done
    skip High Network Congestion (Gradual Buildup): no reliable effect in this fold
seed=7 fold=3 done
seed=7 fold=4 done
seed=7 fold=5 done

=== Per-class F1 (weakest first) ===
                                    class  support  F1_real_only  F1_real_v3  delta
High Network Congestion (Gradual Buildup)       56         0.000       0.000  0.000
                                  Jamming      185         0.087       0.000 -0.087
 Faulty Handover Algorithm (Too Frequent)       45         0.159       0.136 -0.023
             Faulty RF Filters (Temporal)       67         0.182       0.227  0.046
                 Resource Allocation Bugs       43         0.126       0.235  0.109
   High Network Congestion (Sudden Spike)       37         0.041       0.299  0.258
         Co

In [29]:
# =============================================================
# CELL 25 - Fix 1: add Jamming to v3 generation (nested CV, 2 seeds)
# =============================================================
JAM = "Jamming"

# --- 1) Add training Jamming windows to the signal cache ---
jam_rows = si_train[train_y_str == JAM]
missing = [int(r) for r in jam_rows if int(r) not in ROW2IDX]
if missing:
    cache_sig = np.concatenate([cache_sig, np.stack([window_signals(r) for r in missing])])
    for r in missing:
        ROW2IDX[r] = len(ROW2IDX)

s_j, e_j = meta.loc[jam_rows, "anom_start"].values, meta.loc[jam_rows, "anom_end"].values
partial = (s_j >= 0) & ((128 - (e_j - s_j + 1)) >= MIN_OUTSIDE)
print(f"Jamming training windows: {len(jam_rows)} | sessions: {np.unique(session_id_by_row[jam_rows])} "
      f"| with a normal part (within-window usable): {partial.sum()}")
print(context.loc[jam_rows, ["zone", "app"]].value_counts().to_string(), "\n")

# --- 2) Effects for Jamming: channels selected via context-matched comparison ---
def estimate_effects_jam(train_rows, hosts):
    hz, ha = context["zone"].values[hosts], context["app"].values[hosts]
    def ref_of(rows):
        xs = np.stack([sig(r) for r in rows])
        return np.median(xs.mean(2), 0), np.median(xs.std(2), 0)
    REF_f  = {(z, a): ref_of(hosts[(hz == z) & (ha == a)]) for z, a in sorted(set(zip(hz, ha)))}
    REFZ_f = {z: ref_of(hosts[hz == z]) for z in sorted(set(hz))}

    rows = train_rows[LABELS[train_rows] == JAM]
    rec, donors = {k: [] for k in KEYS}, []
    for r in rows:
        s, e = seg_bounds(r)
        if (128 - (e - s + 1)) >= MIN_OUTSIDE:
            vals = compute_effect(sig(r), s, e)                         # within-window
        else:
            z, a = context.loc[r, "zone"], context.loc[r, "app"]
            ref = REF_f.get((z, a), REFZ_f.get(z))
            if ref is None:
                continue
            vals = compute_effect(sig(r), s, e, ref)                    # context-matched
        for k, v in zip(KEYS, vals):
            rec[k].append(v)
        donors.append(r)
    rec = {k: np.array(v) for k, v in rec.items()}

    affected = np.zeros(16, bool)
    for c in range(16):                                                 # gate on ALL Jamming donors
        for arr in (rec["diff"][:, c], rec["log_std_ratio"][:, c], rec["slope"][:, c]):
            v = arr[np.isfinite(arr)]
            if len(v) < 10 or np.allclose(v, 0):
                continue
            try:
                if wilcoxon(v).pvalue < ALPHA:
                    affected[c] = True
                    break
            except ValueError:
                pass
    return {JAM: {**rec, "affected": affected, "rows_used": np.array(donors)}}

# --- 3) Generate Jamming only (same injection as v3) ---
def generate_jam(eff, train_rows, hosts, rng, ch_min, ch_max):
    hz, ha = context["zone"].values[hosts], context["app"].values[hosts]
    E = eff[JAM]
    X, Y = [], []
    if E["affected"].sum() == 0 or len(E["rows_used"]) == 0:
        return np.empty((0, 64), np.float32), np.array([])
    f_rows = train_rows[LABELS[train_rows] == JAM]
    f_sess = session_id_by_row[f_rows]
    for _ in range(max(0, N_TARGET - len(f_rows))):
        k = rng.integers(len(E["rows_used"]))
        donor = E["rows_used"][k]
        same = f_rows[f_sess == session_id_by_row[donor]]
        s, e = seg_bounds(same[rng.integers(len(same))])
        z, a = context.loc[donor, "zone"], context.loc[donor, "app"]
        pool = hosts[(hz == z) & (ha == a)]
        if len(pool) == 0:
            pool = hosts[hz == z]
        if len(pool) == 0:
            pool = hosts
        x = sig(pool[rng.integers(len(pool))]).copy()
        tc = np.arange(e - s + 1) - (e - s) / 2.0
        for c, ch in enumerate(NUMERICAL_CHANNELS):
            if not E["affected"][c]:
                continue
            seg = x[c, s:e + 1]
            m, sd = seg.mean(), seg.std()
            lr, lsr, slp = E["log_ratio"][k, c], E["log_std_ratio"][k, c], E["slope"][k, c]
            new_m = m * np.exp(lr) if (ch not in ADDITIVE and np.isfinite(lr) and m > LEVEL_FLOOR[c]) \
                    else m + E["diff"][k, c]
            if sd > STD_FLOOR[c] and np.isfinite(lsr):
                dev = (seg - m) * np.exp(lsr)
            elif sd <= STD_FLOOR[c]:
                d_sd = E["seg_std"][k, c]
                if ch not in ADDITIVE and E["seg_mean"][k, c] > LEVEL_FLOOR[c]:
                    d_sd = d_sd / E["seg_mean"][k, c] * max(abs(new_m), LEVEL_FLOOR[c])
                dev = rng.normal(0, d_sd, len(seg))
            else:
                dev = seg - m
            trend = 0.0
            if np.isfinite(slp):
                trend = slp * (1.0 if ch in ADDITIVE else max(abs(m), LEVEL_FLOOR[c])) * tc
            x[c, s:e + 1] = np.clip(clip_channel(new_m + dev + trend, ch), ch_min[c], ch_max[c])
        X.append(array_to_config_A(x)); Y.append(JAM)
    return np.array(X, dtype=np.float32), np.array(Y)

# --- 4) Nested CV: real only vs v3 (10 classes) vs v3 + Jamming ---
N_TARGET = 500
oofJ = {(m, s): np.empty(len(real_y), dtype=object) for m in ("base", "v3", "v3+jam") for s in BD_SEEDS}
for seed in BD_SEEDS:
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for f_i, (tr, va) in enumerate(cv.split(real_X, real_y, groups=real_sess)):
        val_sess = set(real_sess[va].tolist())
        tr_rows  = real_rows[tr]
        hosts_ok = host_rows[~np.isin(host_sess, list(val_sess))]
        ch_min = np.minimum(NORMAL_MIN, real_X[tr][:, 2::4].min(0))
        ch_max = np.maximum(NORMAL_MAX, real_X[tr][:, 3::4].max(0))

        eff10 = estimate_effects(tr_rows, hosts_ok)
        X10, Y10 = generate(eff10, tr_rows, hosts_ok, np.random.default_rng(seed * 100 + f_i), ch_min, ch_max)
        effJ = estimate_effects_jam(tr_rows, hosts_ok)
        XJ, YJ = generate_jam(effJ, tr_rows, hosts_ok, np.random.default_rng(seed * 100 + f_i + 50), ch_min, ch_max)

        Xtr, ytr = real_X[tr], real_y[tr]
        ones = lambda n: np.ones(n)
        oofJ[("base", seed)][va] = fit_predict_w(Xtr, ytr, ones(len(ytr)), real_X[va], seed)
        oofJ[("v3", seed)][va] = fit_predict_w(np.vstack([Xtr, X10]), np.concatenate([ytr, Y10]),
                                                ones(len(ytr) + len(Y10)), real_X[va], seed)
        Xa = np.vstack([Xtr, X10] + ([XJ] if len(YJ) else []))
        ya = np.concatenate([ytr, Y10] + ([YJ] if len(YJ) else []))
        oofJ[("v3+jam", seed)][va] = fit_predict_w(Xa, ya, ones(len(ya)), real_X[va], seed)

        jam_ch = [NUMERICAL_CHANNELS[c] for c in np.where(effJ[JAM]["affected"])[0]]
        print(f"seed={seed} fold={f_i+1} done | synthetic Jamming={len(YJ)} | affected: {jam_ch}")
N_TARGET = 150

# --- 5) Results ---
res_tab = pd.DataFrame({"class": classes})
for m in ("base", "v3", "v3+jam"):
    res_tab[m] = np.mean([f1_per_class(oofJ[(m, s)])[0] for s in BD_SEEDS], axis=0)
res_tab["support"] = support
print("\n=== Per-class F1 ===")
print(res_tab.sort_values("v3+jam").round(3).to_string(index=False))
print(f"\nMacro F1 (pooled):  real only = {res_tab['base'].mean():.4f} | "
      f"v3 = {res_tab['v3'].mean():.4f} | v3 + Jamming = {res_tab['v3+jam'].mean():.4f}")
res_tab.to_csv(os.path.join(SYN_DIR, "fix1_jamming_results.csv"), index=False)
print("Saved fix1_jamming_results.csv")

Jamming training windows: 185 | sessions: [0 2] | with a normal part (within-window usable): 0
zone  app    
A     File       93
      Youtube    92 

seed=42 fold=1 done | synthetic Jamming=408 | affected: ['RSRP', 'DL_BLER', 'DL_MCS', 'UL_BLER', 'UL_MCS', 'UL_NPRB', 'UL_SNR', 'TX_Bytes', 'RX_Bytes', 'Estimated_UL_Buffer', 'PRBs_DL_Current', 'PRBs_UL_Current', 'PRB_Utilization_DL', 'PRB_Utilization_UL', 'UL_NumberOfPackets', 'DL_NumberOfPackets']
seed=42 fold=2 done | synthetic Jamming=407 | affected: ['RSRP', 'DL_BLER', 'DL_MCS', 'UL_BLER', 'UL_MCS', 'UL_NPRB', 'UL_SNR', 'TX_Bytes', 'RX_Bytes', 'Estimated_UL_Buffer', 'PRBs_DL_Current', 'PRBs_UL_Current', 'PRB_Utilization_DL', 'PRB_Utilization_UL', 'UL_NumberOfPackets', 'DL_NumberOfPackets']
    skip High Network Congestion (Gradual Buildup): no reliable effect in this fold
seed=42 fold=3 done | synthetic Jamming=315 | affected: ['RSRP', 'DL_BLER', 'DL_MCS', 'UL_BLER', 'UL_MCS', 'UL_NPRB', 'UL_SNR', 'TX_Bytes', 'RX_Bytes', 'Estimated_

In [30]:
# =============================================================
# CELL 26 - Fix 1b: Jamming with dataset-provided affected KPIs
#           + injection across all apps in the same zone
# =============================================================
def jam_affected_from_meta(train_rows):
    """Channels listed in the dataset's affected_kpis for >= half of the training Jamming windows."""
    rows = train_rows[LABELS[train_rows] == JAM]
    counts = Counter()
    for s in meta.loc[rows, "affected_kpis"].fillna(""):
        counts.update([k for k in str(s).split(";") if k])
    keep = {k for k, n in counts.items() if n >= 0.5 * len(rows)}
    return np.array([ch in keep for ch in NUMERICAL_CHANNELS]), counts

def generate_jam_xapp(eff, train_rows, hosts, rng, ch_min, ch_max):
    """Same v3 injection, but hosts are drawn from ALL apps in the donor's zone."""
    hz, ha = context["zone"].values[hosts], context["app"].values[hosts]
    E = eff[JAM]
    X, Y = [], []
    if E["affected"].sum() == 0 or len(E["rows_used"]) == 0:
        return np.empty((0, 64), np.float32), np.array([])
    f_rows = train_rows[LABELS[train_rows] == JAM]
    f_sess = session_id_by_row[f_rows]
    for _ in range(max(0, N_TARGET - len(f_rows))):
        k = rng.integers(len(E["rows_used"]))
        donor = E["rows_used"][k]
        same = f_rows[f_sess == session_id_by_row[donor]]
        s, e = seg_bounds(same[rng.integers(len(same))])
        z = context.loc[donor, "zone"]
        apps = np.unique(ha[hz == z])
        app = apps[rng.integers(len(apps))]                   # pick an app uniformly
        pool = hosts[(hz == z) & (ha == app)]
        if len(pool) == 0:
            pool = hosts[hz == z] if (hz == z).any() else hosts
        x = sig(pool[rng.integers(len(pool))]).copy()
        tc = np.arange(e - s + 1) - (e - s) / 2.0
        for c, ch in enumerate(NUMERICAL_CHANNELS):
            if not E["affected"][c]:
                continue
            seg = x[c, s:e + 1]
            m, sd = seg.mean(), seg.std()
            lr, lsr, slp = E["log_ratio"][k, c], E["log_std_ratio"][k, c], E["slope"][k, c]
            new_m = m * np.exp(lr) if (ch not in ADDITIVE and np.isfinite(lr) and m > LEVEL_FLOOR[c]) \
                    else m + E["diff"][k, c]
            if sd > STD_FLOOR[c] and np.isfinite(lsr):
                dev = (seg - m) * np.exp(lsr)
            elif sd <= STD_FLOOR[c]:
                d_sd = E["seg_std"][k, c]
                if ch not in ADDITIVE and E["seg_mean"][k, c] > LEVEL_FLOOR[c]:
                    d_sd = d_sd / E["seg_mean"][k, c] * max(abs(new_m), LEVEL_FLOOR[c])
                dev = rng.normal(0, d_sd, len(seg))
            else:
                dev = seg - m
            trend = 0.0
            if np.isfinite(slp):
                trend = slp * (1.0 if ch in ADDITIVE else max(abs(m), LEVEL_FLOOR[c])) * tc
            x[c, s:e + 1] = np.clip(clip_channel(new_m + dev + trend, ch), ch_min[c], ch_max[c])
        X.append(array_to_config_A(x)); Y.append(JAM)
    return np.array(X, dtype=np.float32), np.array(Y)

# --- Nested CV: only the new model is trained; real-only and v3 reused from Cell 25 ---
N_TARGET = 500
for s_ in BD_SEEDS:
    oofJ[("v3+jam1b", s_)] = np.empty(len(real_y), dtype=object)

for seed in BD_SEEDS:
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for f_i, (tr, va) in enumerate(cv.split(real_X, real_y, groups=real_sess)):
        val_sess = set(real_sess[va].tolist())
        tr_rows  = real_rows[tr]
        hosts_ok = host_rows[~np.isin(host_sess, list(val_sess))]
        ch_min = np.minimum(NORMAL_MIN, real_X[tr][:, 2::4].min(0))
        ch_max = np.maximum(NORMAL_MAX, real_X[tr][:, 3::4].max(0))

        eff10 = estimate_effects(tr_rows, hosts_ok)
        X10, Y10 = generate(eff10, tr_rows, hosts_ok, np.random.default_rng(seed * 100 + f_i), ch_min, ch_max)

        effJ = estimate_effects_jam(tr_rows, hosts_ok)
        aff, counts = jam_affected_from_meta(tr_rows)
        effJ[JAM]["affected"] = aff                        # dataset-provided channels
        XJ, YJ = generate_jam_xapp(effJ, tr_rows, hosts_ok,
                                   np.random.default_rng(seed * 100 + f_i + 50), ch_min, ch_max)

        Xtr, ytr = real_X[tr], real_y[tr]
        Xa = np.vstack([Xtr, X10] + ([XJ] if len(YJ) else []))
        ya = np.concatenate([ytr, Y10] + ([YJ] if len(YJ) else []))
        oofJ[("v3+jam1b", seed)][va] = fit_predict_w(Xa, ya, np.ones(len(ya)), real_X[va], seed)

        print(f"seed={seed} fold={f_i+1} done | synthetic Jamming={len(YJ)} | "
              f"affected: {[NUMERICAL_CHANNELS[c] for c in np.where(aff)[0]]}")
N_TARGET = 150

print("\naffected_kpis counts (last fold):", dict(counts))

# --- Results ---
for m in ("v3+jam1b",):
    res_tab[m] = np.mean([f1_per_class(oofJ[(m, s)])[0] for s in BD_SEEDS], axis=0)
print("\n=== Per-class F1 ===")
print(res_tab.sort_values("v3+jam1b").round(3).to_string(index=False))
print(f"\nMacro F1 (pooled): real only = {res_tab['base'].mean():.4f} | v3 = {res_tab['v3'].mean():.4f} | "
      f"v3+jam = {res_tab['v3+jam'].mean():.4f} | v3+jam1b = {res_tab['v3+jam1b'].mean():.4f}")
res_tab.to_csv(os.path.join(SYN_DIR, "fix1b_jamming_results.csv"), index=False)
print("Saved fix1b_jamming_results.csv")

seed=42 fold=1 done | synthetic Jamming=408 | affected: ['DL_BLER', 'UL_BLER', 'UL_MCS', 'TX_Bytes', 'RX_Bytes']
seed=42 fold=2 done | synthetic Jamming=407 | affected: ['DL_BLER', 'UL_BLER', 'UL_MCS', 'TX_Bytes', 'RX_Bytes']
    skip High Network Congestion (Gradual Buildup): no reliable effect in this fold
seed=42 fold=3 done | synthetic Jamming=315 | affected: ['DL_BLER', 'UL_BLER', 'UL_MCS', 'TX_Bytes', 'RX_Bytes']
seed=42 fold=4 done | synthetic Jamming=315 | affected: ['DL_BLER', 'UL_BLER', 'UL_MCS', 'TX_Bytes', 'RX_Bytes']
seed=42 fold=5 done | synthetic Jamming=315 | affected: ['DL_BLER', 'UL_BLER', 'UL_MCS', 'TX_Bytes', 'RX_Bytes']
seed=7 fold=1 done | synthetic Jamming=408 | affected: ['DL_BLER', 'UL_BLER', 'UL_MCS', 'TX_Bytes', 'RX_Bytes']
seed=7 fold=2 done | synthetic Jamming=407 | affected: ['DL_BLER', 'UL_BLER', 'UL_MCS', 'TX_Bytes', 'RX_Bytes']
    skip High Network Congestion (Gradual Buildup): no reliable effect in this fold
seed=7 fold=3 done | synthetic Jamming=315 

In [31]:
# =============================================================
# CELL 27 - Final confirmation: 5 seeds x 5 folds, nested, checkpointed
# =============================================================
CONF_SEEDS = [42, 7, 123, 2024, 99]
CKPT27 = os.path.join(SYN_DIR, "final_confirm_partial.csv")

rows27, done27 = [], set()
if os.path.exists(CKPT27):
    prev27 = pd.read_csv(CKPT27)
    rows27 = prev27.to_dict("records")
    done27 = set(zip(prev27["seed"], prev27["fold"]))
    print(f"Resuming: {len(done27)} fold(s) already done")

macro = lambda y_true, y_pred: f1_score(y_true, y_pred, average="macro", zero_division=0)

N_TARGET = 500
for seed in CONF_SEEDS:
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for f_i, (tr, va) in enumerate(cv.split(real_X, real_y, groups=real_sess)):
        if (seed, f_i + 1) in done27:
            continue
        val_sess = set(real_sess[va].tolist())
        tr_rows  = real_rows[tr]
        hosts_ok = host_rows[~np.isin(host_sess, list(val_sess))]
        ch_min = np.minimum(NORMAL_MIN, real_X[tr][:, 2::4].min(0))
        ch_max = np.maximum(NORMAL_MAX, real_X[tr][:, 3::4].max(0))

        # 10 synthetic-fault classes (v3)
        eff10 = estimate_effects(tr_rows, hosts_ok)
        X10, Y10 = generate(eff10, tr_rows, hosts_ok, np.random.default_rng(seed * 100 + f_i), ch_min, ch_max)
        # Jamming (fix 1b)
        effJ = estimate_effects_jam(tr_rows, hosts_ok)
        effJ[JAM]["affected"] = jam_affected_from_meta(tr_rows)[0]
        XJ, YJ = generate_jam_xapp(effJ, tr_rows, hosts_ok,
                                   np.random.default_rng(seed * 100 + f_i + 50), ch_min, ch_max)

        Xtr, ytr, Xva, yva = real_X[tr], real_y[tr], real_X[va], real_y[va]
        p_base = fit_predict_w(Xtr, ytr, np.ones(len(ytr)), Xva, seed)
        X_v3, y_v3 = np.vstack([Xtr, X10]), np.concatenate([ytr, Y10])
        p_v3 = fit_predict_w(X_v3, y_v3, np.ones(len(y_v3)), Xva, seed)
        X_f = np.vstack([X_v3] + ([XJ] if len(YJ) else []))
        y_f = np.concatenate([y_v3] + ([YJ] if len(YJ) else []))
        p_final = fit_predict_w(X_f, y_f, np.ones(len(y_f)), Xva, seed)

        rows27.append({"seed": seed, "fold": f_i + 1,
                       "base": macro(yva, p_base), "v3": macro(yva, p_v3), "final": macro(yva, p_final),
                       "n_train": int(len(tr)), "n_val": int(len(va))})
        pd.DataFrame(rows27).to_csv(CKPT27, index=False)
        r = rows27[-1]
        print(f"seed={seed:<5} fold={f_i+1}  real={r['base']:.4f}  v3={r['v3']:.4f}  "
              f"final={r['final']:.4f}  (saved)")
N_TARGET = 150

# ---------------- Statistics ----------------
R = pd.DataFrame(rows27)
ratio = np.mean(R["n_val"] / R["n_train"])

def paired(a, b):
    d = (R[a] - R[b]).values
    t_stat = d.mean() / np.sqrt((1 / len(d) + ratio) * d.var(ddof=1))
    p = 2 * (1 - t_dist.cdf(abs(t_stat), df=len(d) - 1))
    return d.mean(), int((d > 0).sum()), len(d), p

print("\n" + "=" * 64)
for m, name in [("base", "Real only"), ("v3", "v3 (10 classes)"), ("final", "Final (v3 + Jamming)")]:
    print(f"{name:<24}: {R[m].mean():.4f} ± {R[m].std():.4f}")
for a, b, name in [("final", "base", "Final vs Real only"), ("final", "v3", "Final vs v3")]:
    dm, w, n, p = paired(a, b)
    print(f"{name:<24}: Δ={dm:+.4f}  wins {w}/{n}  corrected t-test p={p:.4f}")
print("=" * 64)

fl = R.groupby("fold")[["base", "v3", "final"]].mean()
fl["Δ final-real"] = fl["final"] - fl["base"]
print("\nFold-level (mean over 5 seeds):")
print(fl.round(4).to_string())

R.to_csv(os.path.join(SYN_DIR, "final_confirm_records.csv"), index=False)
print("\nSaved final_confirm_records.csv")

seed=42    fold=1  real=0.2132  v3=0.3254  final=0.3288  (saved)
seed=42    fold=2  real=0.2749  v3=0.3332  final=0.4246  (saved)
    skip High Network Congestion (Gradual Buildup): no reliable effect in this fold
seed=42    fold=3  real=0.2909  v3=0.3753  final=0.3305  (saved)
seed=42    fold=4  real=0.2429  v3=0.2677  final=0.2242  (saved)
seed=42    fold=5  real=0.4491  v3=0.4118  final=0.4041  (saved)
seed=7     fold=1  real=0.2350  v3=0.3420  final=0.3369  (saved)
seed=7     fold=2  real=0.2856  v3=0.3528  final=0.3558  (saved)
    skip High Network Congestion (Gradual Buildup): no reliable effect in this fold
seed=7     fold=3  real=0.2932  v3=0.3652  final=0.3254  (saved)
seed=7     fold=4  real=0.2143  v3=0.2385  final=0.2313  (saved)
seed=7     fold=5  real=0.4651  v3=0.4689  final=0.4598  (saved)
seed=123   fold=1  real=0.1937  v3=0.3005  final=0.3055  (saved)
seed=123   fold=2  real=0.2437  v3=0.3686  final=0.4269  (saved)
    skip High Network Congestion (Gradual Buildup): 

In [32]:
print(paired("v3", "base"))

(np.float64(0.059590534427353195), 23, 25, np.float64(0.03028291273644368))


In [33]:
# =============================================================
# CELL 28 - v3 + FFT features (5 seeds x 5 folds, nested, checkpointed)
# =============================================================
CKPT28 = os.path.join(SYN_DIR, "fft_v3_partial.csv")

def fft_feats(x):
    """(16, 128) -> (80,): per channel, first 4 FFT band magnitudes + total oscillation energy."""
    xc = x - x.mean(axis=1, keepdims=True)                 # remove level (Config A has it)
    mag = np.abs(np.fft.rfft(xc, axis=1))
    return np.concatenate([mag[:, 1:5], mag[:, 1:].sum(1, keepdims=True)], axis=1).reshape(-1)

def real_sig(r):
    return sig(r) if int(r) in ROW2IDX else window_signals(r)

# FFT for the real fault windows (same order as real_X)
real_FFT = np.stack([fft_feats(real_sig(r)) for r in real_rows]).astype(np.float32)
real_XF = np.hstack([real_X, real_FFT])                     # 64 + 80 = 144
print("Real features with FFT:", real_XF.shape)

# Make generate() output Config A + FFT (144) for synthetic windows
if "_CFGA_ORIG" not in globals():
    _CFGA_ORIG = array_to_config_A
def _cfgA_fft(x):
    return np.concatenate([_CFGA_ORIG(x), fft_feats(x).astype(np.float32)])

rows28, done28 = [], set()
if os.path.exists(CKPT28):
    prev28 = pd.read_csv(CKPT28)
    rows28 = prev28.to_dict("records")
    done28 = set(zip(prev28["seed"], prev28["fold"]))
    print(f"Resuming: {len(done28)} fold(s) already done")

oof_fft = {s: np.empty(len(real_y), dtype=object) for s in CONF_SEEDS}

N_TARGET = 500
array_to_config_A = _cfgA_fft
try:
    for seed in CONF_SEEDS:
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        for f_i, (tr, va) in enumerate(cv.split(real_X, real_y, groups=real_sess)):
            if (seed, f_i + 1) in done28:
                continue
            val_sess = set(real_sess[va].tolist())
            tr_rows  = real_rows[tr]
            hosts_ok = host_rows[~np.isin(host_sess, list(val_sess))]
            ch_min = np.minimum(NORMAL_MIN, real_X[tr][:, 2::4].min(0))
            ch_max = np.maximum(NORMAL_MAX, real_X[tr][:, 3::4].max(0))

            eff10 = estimate_effects(tr_rows, hosts_ok)
            X10, Y10 = generate(eff10, tr_rows, hosts_ok,
                                np.random.default_rng(seed * 100 + f_i), ch_min, ch_max)   # 144 features

            Xtr, ytr, Xva, yva = real_XF[tr], real_y[tr], real_XF[va], real_y[va]
            p_rfft = fit_predict_w(Xtr, ytr, np.ones(len(ytr)), Xva, seed)
            Xa, ya = np.vstack([Xtr, X10]), np.concatenate([ytr, Y10])
            p_vfft = fit_predict_w(Xa, ya, np.ones(len(ya)), Xva, seed)
            oof_fft[seed][va] = p_vfft

            rows28.append({"seed": seed, "fold": f_i + 1,
                           "real_fft": macro(yva, p_rfft), "v3_fft": macro(yva, p_vfft)})
            pd.DataFrame(rows28).to_csv(CKPT28, index=False)
            print(f"seed={seed:<5} fold={f_i+1}  real+FFT={rows28[-1]['real_fft']:.4f}  "
                  f"v3+FFT={rows28[-1]['v3_fft']:.4f}  (saved)")
finally:
    array_to_config_A = _CFGA_ORIG          # always restore
    N_TARGET = 150

# ---------------- Statistics ----------------
R27 = pd.read_csv(os.path.join(SYN_DIR, "final_confirm_records.csv"))
R = R27.merge(pd.DataFrame(rows28), on=["seed", "fold"])

print("\n" + "=" * 64)
for m, name in [("base", "Real only"), ("real_fft", "Real + FFT"),
                ("v3", "v3"), ("v3_fft", "v3 + FFT")]:
    print(f"{name:<14}: {R[m].mean():.4f} ± {R[m].std():.4f}")
for a, b, name in [("v3_fft", "v3", "v3+FFT vs v3"),
                   ("v3_fft", "base", "v3+FFT vs Real"),
                   ("real_fft", "base", "Real+FFT vs Real")]:
    dm, w, n, p = paired(a, b)
    print(f"{name:<18}: Δ={dm:+.4f}  wins {w}/{n}  corrected t-test p={p:.4f}")
print("=" * 64)

fl = R.groupby("fold")[["base", "real_fft", "v3", "v3_fft"]].mean()
fl["Δ v3fft-v3"] = fl["v3_fft"] - fl["v3"]
print("\nFold-level (mean over seeds):")
print(fl.round(4).to_string())

# Per-class check (seeds 42 & 7, comparable to Cell 24)
f1_vfft = np.mean([f1_per_class(oof_fft[s])[0] for s in (42, 7) if (oof_fft[s] != None).all()], axis=0)
pc = pd.read_csv(os.path.join(SYN_DIR, "per_class_breakdown.csv")).set_index("class")
pc["F1_v3_fft"] = pd.Series(f1_vfft, index=classes)
print("\nPer-class F1 (seeds 42 & 7):")
print(pc[["support", "F1_real_only", "F1_real_v3", "F1_v3_fft"]].sort_values("F1_v3_fft").round(3).to_string())

R.to_csv(os.path.join(SYN_DIR, "fft_v3_records.csv"), index=False)
print("\nSaved fft_v3_records.csv")

Real features with FFT: (935, 144)
seed=42    fold=1  real+FFT=0.3212  v3+FFT=0.3326  (saved)
seed=42    fold=2  real+FFT=0.3049  v3+FFT=0.3550  (saved)
    skip High Network Congestion (Gradual Buildup): no reliable effect in this fold
seed=42    fold=3  real+FFT=0.3451  v3+FFT=0.3963  (saved)
seed=42    fold=4  real+FFT=0.2170  v3+FFT=0.3034  (saved)
seed=42    fold=5  real+FFT=0.5485  v3+FFT=0.4829  (saved)
seed=7     fold=1  real+FFT=0.3283  v3+FFT=0.3318  (saved)
seed=7     fold=2  real+FFT=0.2988  v3+FFT=0.3091  (saved)
    skip High Network Congestion (Gradual Buildup): no reliable effect in this fold
seed=7     fold=3  real+FFT=0.3295  v3+FFT=0.3857  (saved)
seed=7     fold=4  real+FFT=0.2474  v3+FFT=0.2690  (saved)
seed=7     fold=5  real+FFT=0.5518  v3+FFT=0.4877  (saved)
seed=123   fold=1  real+FFT=0.3034  v3+FFT=0.3263  (saved)
seed=123   fold=2  real+FFT=0.2664  v3+FFT=0.3322  (saved)
    skip High Network Congestion (Gradual Buildup): no reliable effect in this fold
seed=

In [19]:
# =============================================================
# CELL 29 - v3 + Config B / Config C (numeric) + FFT
#           5 seeds x 5 folds, nested, checkpointed
# =============================================================
CKPT29 = os.path.join(SYN_DIR, "configBC_partial.csv")
if "N_THREADS" not in globals():
    N_THREADS = max(1, (os.cpu_count() or 4) - 2)
if "_CFGA_ORIG" not in globals():
    _CFGA_ORIG = array_to_config_A

# --- Feature functions (all computed from raw 16 x 128 signals) ---
def fft_feats(x):
    xc = x - x.mean(axis=1, keepdims=True)
    mag = np.abs(np.fft.rfft(xc, axis=1))
    return np.concatenate([mag[:, 1:5], mag[:, 1:].sum(1, keepdims=True)], axis=1).reshape(-1)

def patch_stats(x, n=8):
    """Split time into 8 patches; mean + std per patch per channel -> 16 x 8 x 2 = 256."""
    parts = np.array_split(x, n, axis=1)
    return np.concatenate([np.stack([p.mean(1), p.std(1)], axis=1) for p in parts], axis=1).reshape(-1)

def diff_stats(x, n=8):
    """Same patch stats on the first-order differences (rate of change) -> 256."""
    return patch_stats(np.diff(x, axis=1), n)

def full_feats(x):
    """[Config A 64 | patchwise 256 | differences 256 | FFT 80] = 656"""
    return np.concatenate([_CFGA_ORIG(x), patch_stats(x), diff_stats(x), fft_feats(x)]).astype(np.float32)

IDX_B_FFT = np.r_[0:320, 576:656]      # Config B (A + patchwise) + FFT = 400
IDX_C_FFT = np.arange(656)             # Config C numeric (B + differences) + FFT = 656

def real_sig(r):
    return sig(r) if int(r) in ROW2IDX else window_signals(r)

def fit_predict_w(X_tr, y_tr, w_tr, X_va, seed):
    classes_ = np.unique(y_tr)
    eff_ = {c: w_tr[y_tr == c].sum() for c in classes_}
    total = sum(eff_.values())
    cw = {c: total / (len(classes_) * eff_[c]) for c in classes_}
    sw = w_tr * np.array([cw[c] for c in y_tr])
    clf = CatBoostClassifier(random_seed=seed, verbose=0, thread_count=N_THREADS)
    clf.fit(X_tr, y_tr, sample_weight=sw)
    return clf.predict(X_va).ravel()

macro = lambda yt, yp: f1_score(yt, yp, average="macro", zero_division=0)

# --- Real windows: all features from raw signals ---
real_full = np.stack([full_feats(real_sig(r)) for r in real_rows])
print("Real features:", real_full.shape,
      "| Config A check (max abs diff vs train_X):", float(np.abs(real_full[:, :64] - real_X).max()))

# --- Resume ---
rows29, done29 = [], set()
if os.path.exists(CKPT29):
    prev29 = pd.read_csv(CKPT29)
    rows29 = prev29.to_dict("records")
    done29 = set(zip(prev29["seed"], prev29["fold"]))
    print(f"Resuming: {len(done29)} fold(s) already done")

CONF_SEEDS = [42, 7, 123, 2024, 99]
N_TARGET = 500
array_to_config_A = full_feats            # generate() now outputs all 656 features
try:
    for seed in CONF_SEEDS:
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        for f_i, (tr, va) in enumerate(cv.split(real_X, real_y, groups=real_sess)):
            if (seed, f_i + 1) in done29:
                continue
            val_sess = set(real_sess[va].tolist())
            tr_rows  = real_rows[tr]
            hosts_ok = host_rows[~np.isin(host_sess, list(val_sess))]
            ch_min = np.minimum(NORMAL_MIN, real_X[tr][:, 2::4].min(0))
            ch_max = np.maximum(NORMAL_MAX, real_X[tr][:, 3::4].max(0))

            eff10 = estimate_effects(tr_rows, hosts_ok)
            Xs, Ys = generate(eff10, tr_rows, hosts_ok,
                              np.random.default_rng(seed * 100 + f_i), ch_min, ch_max)

            rec = {"seed": seed, "fold": f_i + 1}
            for name, idx in (("B_fft", IDX_B_FFT), ("C_fft", IDX_C_FFT)):
                Xa = np.vstack([real_full[tr][:, idx], Xs[:, idx]])
                ya = np.concatenate([real_y[tr], Ys])
                p = fit_predict_w(Xa, ya, np.ones(len(ya)), real_full[va][:, idx], seed)
                rec[name] = macro(real_y[va], p)
            rows29.append(rec)
            pd.DataFrame(rows29).to_csv(CKPT29, index=False)
            print(f"seed={seed:<5} fold={f_i+1}  B+FFT={rec['B_fft']:.4f}  C+FFT={rec['C_fft']:.4f}  (saved)")
finally:
    array_to_config_A = _CFGA_ORIG
    N_TARGET = 150

# ---------------- Statistics ----------------
R = pd.read_csv(os.path.join(SYN_DIR, "fft_v3_records.csv")).merge(pd.DataFrame(rows29), on=["seed", "fold"])
ratio = np.mean(R["n_val"] / R["n_train"])

def paired2(a, b):
    d = (R[a] - R[b]).values
    t_stat = d.mean() / np.sqrt((1 / len(d) + ratio) * d.var(ddof=1))
    return d.mean(), int((d > 0).sum()), len(d), 2 * (1 - t_dist.cdf(abs(t_stat), df=len(d) - 1))

print("\n" + "=" * 64)
for m, name in [("base", "Real only (A)"), ("v3", "v3 (A)"), ("v3_fft", "v3 (A + FFT)"),
                ("B_fft", "v3 (B + FFT)"), ("C_fft", "v3 (C + FFT)")]:
    print(f"{name:<16}: {R[m].mean():.4f} ± {R[m].std():.4f}")
for a, b, name in [("B_fft", "v3_fft", "B+FFT vs A+FFT"), ("C_fft", "v3_fft", "C+FFT vs A+FFT"),
                   ("B_fft", "base", "B+FFT vs Real"), ("C_fft", "base", "C+FFT vs Real")]:
    dm, w, n, p = paired2(a, b)
    print(f"{name:<16}: Δ={dm:+.4f}  wins {w}/{n}  corrected t-test p={p:.4f}")
print("=" * 64)
print("\nFold-level (mean over seeds):")
print(R.groupby("fold")[["base", "v3_fft", "B_fft", "C_fft"]].mean().round(4).to_string())

R.to_csv(os.path.join(SYN_DIR, "configBC_records.csv"), index=False)
print("\nSaved configBC_records.csv")

Real features: (935, 656) | Config A check (max abs diff vs train_X): 0.015625
Resuming: 25 fold(s) already done

Real only (A)   : 0.2908 ± 0.0921
v3 (A)          : 0.3504 ± 0.0732
v3 (A + FFT)    : 0.3625 ± 0.0773
v3 (B + FFT)    : 0.3743 ± 0.0960
v3 (C + FFT)    : 0.3812 ± 0.1043
B+FFT vs A+FFT  : Δ=+0.0118  wins 18/25  corrected t-test p=0.4658
C+FFT vs A+FFT  : Δ=+0.0187  wins 17/25  corrected t-test p=0.4386
B+FFT vs Real   : Δ=+0.0835  wins 25/25  corrected t-test p=0.0065
C+FFT vs Real   : Δ=+0.0904  wins 24/25  corrected t-test p=0.0028

Fold-level (mean over seeds):
        base  v3_fft   B_fft   C_fft
fold                                
1     0.2152  0.3409  0.3529  0.3573
2     0.2613  0.3268  0.3257  0.3454
3     0.2926  0.3884  0.4185  0.4057
4     0.2264  0.2688  0.2516  0.2474
5     0.4587  0.4877  0.5228  0.5503

Saved configBC_records.csv


In [20]:
# =============================================================
# CELL 30 - FINAL TEST-SET EVALUATION (run ONCE, do not tune after)
# =============================================================
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# --- Test data: real fault windows from the held-out sessions ---
test_y_all = meta["label"].values[si_test]          # labels from the dataset (verified source)
tmask = test_y_all != NORMAL_LABEL
test_rows, test_y = si_test[tmask], test_y_all[tmask]
print(f"Test fault windows: {len(test_rows)} | sessions: {np.unique(session_id_by_row[test_rows])}")
print(pd.Series(test_y).value_counts().to_string(), "\n")

test_full = np.stack([full_feats(window_signals(r)) for r in test_rows])
if "real_full" not in globals():
    real_full = np.stack([full_feats(real_sig(r)) for r in real_rows])

# --- Effects learned once from ALL training data (test untouched) ---
ch_min = np.minimum(NORMAL_MIN, real_X[:, 2::4].min(0))
ch_max = np.maximum(NORMAL_MAX, real_X[:, 3::4].max(0))
eff_all = estimate_effects(real_rows, host_rows)

FINAL_SEEDS = [42, 7, 123, 2024, 99]
res30, preds30 = [], {}
A = np.arange(64)
cols = {"Real only (A)": A, "v3 (A)": A, "Final (B+FFT)": IDX_B_FFT}

N_TARGET = 500
array_to_config_A = full_feats
try:
    for s in FINAL_SEEDS:
        Xs, Ys = generate(eff_all, real_rows, host_rows, np.random.default_rng(s), ch_min, ch_max)
        models = {
            "Real only (A)": (real_full[:, A], real_y),
            "v3 (A)":        (np.vstack([real_full[:, A], Xs[:, A]]), np.concatenate([real_y, Ys])),
            "Final (B+FFT)": (np.vstack([real_full[:, IDX_B_FFT], Xs[:, IDX_B_FFT]]),
                              np.concatenate([real_y, Ys])),
        }
        for name, (Xtr, ytr) in models.items():
            p = fit_predict_w(Xtr, ytr, np.ones(len(ytr)), test_full[:, cols[name]], s)
            preds30[(name, s)] = p
            res30.append({"model": name, "seed": s,
                          "macro_f1": macro(test_y, p), "accuracy": accuracy_score(test_y, p)})
        print(f"seed={s} done")
finally:
    array_to_config_A = _CFGA_ORIG
    N_TARGET = 150

# --- Results ---
T = pd.DataFrame(res30)
summary30 = T.groupby("model")[["macro_f1", "accuracy"]].agg(["mean", "std"]).round(4)
print("\n=== TEST SET (held-out sessions) ===")
print(summary30.to_string())

test_classes = np.unique(test_y)
def pc_f1(p):
    return precision_recall_fscore_support(test_y, p, labels=test_classes, zero_division=0)[2]

pc = pd.DataFrame({"class": test_classes,
                   "support": [int((test_y == c).sum()) for c in test_classes]})
for name in ["Real only (A)", "v3 (A)", "Final (B+FFT)"]:
    pc[name] = np.mean([pc_f1(preds30[(name, s)]) for s in FINAL_SEEDS], axis=0)
print("\n=== Per-class F1 on the test set ===")
print(pc.sort_values("Final (B+FFT)").round(3).to_string(index=False))

T.to_csv(os.path.join(SYN_DIR, "final_test_records.csv"), index=False)
pc.to_csv(os.path.join(SYN_DIR, "final_test_per_class.csv"), index=False)
print("\nSaved final_test_records.csv and final_test_per_class.csv")

Test fault windows: 300 | sessions: [ 1 13 17]
Jamming                                      94
High Network Congestion (Gradual Buildup)    37
Faulty RF Filters (Temporal)                 33
Co-Channel Interference (Severe)             30
Buffer Overflow (Gradual Buildup)            21
Co-Channel Interference (Mild)               20
Faulty Handover Algorithm (Too Frequent)     19
Resource Allocation Bugs                     17
High Network Congestion (Sudden Spike)       11
Doppler Shift (Severe)                        9
Antenna Failure                               9 

seed=42 done
seed=7 done
seed=123 done
seed=2024 done
seed=99 done

=== TEST SET (held-out sessions) ===
              macro_f1         accuracy        
                  mean     std     mean     std
model                                          
Final (B+FFT)   0.5569  0.0124   0.6800  0.0100
Real only (A)   0.5863  0.0135   0.7053  0.0112
v3 (A)          0.5484  0.0137   0.6720  0.0115

=== Per-class F1 on the test 

In [21]:
np.savez(os.path.join(SYN_DIR, "final_test_preds.npz"),
         test_rows=test_rows, test_y=test_y,
         **{f"{m}__{s}": preds30[(m, s)] for (m, s) in preds30})
print("Saved final_test_preds.npz")

Saved final_test_preds.npz


In [22]:
# =============================================================
# CELL 31 - Explain the test result: bootstrap CI + per-session breakdown
#           (analysis only - no model changes)
# =============================================================
from sklearn.metrics import accuracy_score

models = ["Real only (A)", "v3 (A)", "Final (B+FFT)"]
P = {m: np.stack([preds30[(m, s)] for s in FINAL_SEEDS]) for m in models}   # (5 seeds, 300)
sess_test = session_id_by_row[test_rows]
groups = {s: np.where(sess_test == s)[0] for s in np.unique(sess_test)}

def macro_mean(m, idx):
    return np.mean([f1_score(test_y[idx], P[m][k][idx], average="macro", zero_division=0)
                    for k in range(len(FINAL_SEEDS))])

# --- 1) Bootstrap CI (resample windows within each session, 1,000 times) ---
rng_b = np.random.default_rng(0)
B = 1000
diffs = {"v3 (A)": [], "Final (B+FFT)": []}
for _ in range(B):
    idx = np.concatenate([rng_b.choice(g, size=len(g), replace=True) for g in groups.values()])
    base = macro_mean("Real only (A)", idx)
    for m in diffs:
        diffs[m].append(macro_mean(m, idx) - base)

all_idx = np.arange(len(test_y))
print("=== Bootstrap: difference vs Real only (test macro F1) ===")
for m, d in diffs.items():
    d = np.array(d)
    obs = macro_mean(m, all_idx) - macro_mean("Real only (A)", all_idx)
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"{m:<15}: Δ={obs:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  "
          f"P(worse than real) = {(d < 0).mean():.2f}")

# --- 2) Per-session breakdown ---
rows31 = []
for s, g in groups.items():
    row = {"session": int(s), "n_windows": len(g)}
    for m in models:
        row[f"F1 {m}"] = macro_mean(m, g)
        row[f"Acc {m}"] = np.mean([accuracy_score(test_y[g], P[m][k][g]) for k in range(len(FINAL_SEEDS))])
    rows31.append(row)
S = pd.DataFrame(rows31)
print("\n=== Per-session results ===")
print(S.round(3).to_string(index=False))

print("\n=== Fault classes in each test session ===")
for s, g in groups.items():
    print(f"Session {s}: " + ", ".join(f"{c} ({n})" for c, n in pd.Series(test_y[g]).value_counts().items()))

S.to_csv(os.path.join(SYN_DIR, "test_per_session.csv"), index=False)
pd.DataFrame(diffs).to_csv(os.path.join(SYN_DIR, "test_bootstrap_diffs.csv"), index=False)
print("\nSaved test_per_session.csv and test_bootstrap_diffs.csv")

=== Bootstrap: difference vs Real only (test macro F1) ===
v3 (A)         : Δ=-0.0379  95% CI [-0.0768, -0.0001]  P(worse than real) = 0.98
Final (B+FFT)  : Δ=-0.0294  95% CI [-0.0674, +0.0071]  P(worse than real) = 0.94

=== Per-session results ===
 session  n_windows  F1 Real only (A)  Acc Real only (A)  F1 v3 (A)  Acc v3 (A)  F1 Final (B+FFT)  Acc Final (B+FFT)
       1         94              1.00              1.000      0.665       0.989             0.665              0.991
      13         69              0.13              0.304      0.157       0.275             0.157              0.255
      17        137              0.61              0.705      0.605       0.654             0.650              0.680

=== Fault classes in each test session ===
Session 1: Jamming (94)
Session 13: High Network Congestion (Gradual Buildup) (31), Buffer Overflow (Gradual Buildup) (16), High Network Congestion (Sudden Spike) (11), Co-Channel Interference (Mild) (5), Antenna Failure (3), Faulty Hando